# Dynamic 3D Gaussians (JonathonLuiten/Dynamic3DGaussians): Neural 3D Video multi-view benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/notebooks/04_dynamic3dgaussians_luiten_n3dv.ipynb)

Part of **dynamic-gaussian-splatting-benchmark**, extended to the **multi-view** setting: a controlled comparison of four dynamic Gaussian Splatting methods on the six scenes of the *Neural 3D Video Synthesis from Multi-View Video* (N3DV) dataset, run on a cloud GPU (lightning.ai Studio, Google Colab, or any Linux machine with an NVIDIA card). The method is trained with its official training code; a benchmark monitor attached to its training loop records PSNR, SSIM, LPIPS, evaluation L1, training time, iteration count, number of Gaussians, peak VRAM and model storage into one JSON file per run — the same monitor, the same fields and the same two protocols as the monocular D-NeRF study in notebooks `01`–`03`.

**AI disclosure.** This notebook, the rest of the repository and part of the code used to extract the metrics (the benchmark monitor, the per-scene preparation, the dataset converters and the analysis scripts) were generated with AI assistance (Anthropic's Claude), starting from the code of the official repository of each method. The training code of the method itself is the official one, cloned at run time; the only changes are the monitor hook appended to `train.py`, configuration overrides and environment fixes, all made by the cells below and listed in `docs/METHODOLOGY_MULTIVIEW.md`.

**Method.** Dynamic 3D Gaussians: Tracking by Persistent Dynamic View Synthesis, Luiten et al., 3DV 2024 ([paper](https://arxiv.org/abs/2308.09713), [code](https://github.com/JonathonLuiten/Dynamic3DGaussians)). One set of Gaussians that is optimised **frame by frame**: the first frame is reconstructed from scratch, each later frame starts from the previous one and is optimised under rigidity, rotation, isometry and background-anchoring losses, so the primitives persist and can be tracked.

This method was **studied but not benchmarked** in the monocular part of the project: it requires synchronised multi-camera input by construction. This is its first notebook, written to the same template as `01`–`03`, and it is the one that needed most adaptation — the authors run it on Panoptic Sports, not on N3DV, and the parts of their pipeline that build the dataset and evaluate novel views have not been released. What the notebook adds, and why, is listed in cell 3.5 and in `docs/METHODOLOGY_MULTIVIEW.md`.

Two protocols, unchanged from the monocular study:

* **Protocol A, equal iterations**: every method trains for 30 000 optimisation steps, metrics sampled every 1 000 steps.
* **Protocol B, equal quality**: training stops when the evaluation L1 reaches a per-scene target shared by the four notebooks, metrics sampled every 30 seconds.

**Multi-view conventions** (identical in notebooks `04`–`07`): N3DV at **1352×1014** (half of the native 2704×2028), the **first 50 frames** of every scene, **`cam00` held out** as the test camera, so the test split is exactly 50 views; LPIPS with the VGG backbone from a single backend; evaluation L1 as the common loss. Why each of these was needed is in [`docs/METHODOLOGY_MULTIVIEW.md`](https://github.com/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/docs/METHODOLOGY_MULTIVIEW.md).

**Notebook structure**

*Part 1: configuration, setup and training*

0. User configuration and path resolution
1. Environment and repository setup
2. CUDA module and environment fixes
3. N3DV dataset download and per-scene preparation (shared COLMAP stage, Panoptic-style conversion)
4. Training with incremental benchmarking (single scene or loop, Protocol B calibration, summary)

*Part 2: evaluation and visualisation*

5. Rendering and final metrics
6. Video export of the held-out view

---

# Part 1: configuration, setup and training

This part is safe to run unattended. It configures the run, prepares the environment and the dataset, installs the benchmark instrumentation and trains one scene or a list of scenes. It produces model checkpoints and one JSON summary per run.

**Do not run Part 2 during a training loop.** It renders and computes metrics, which compete for the GPU and would corrupt the training-time and peak-VRAM measurements.

---

## 0. Platform, configuration and paths

Cell 0.0 works out **where this notebook is running** and nothing else. The three
environments it supports need different base directories and different ways of reaching
Google Drive:

| | base directory | Google Drive |
|---|---|---|
| **lightning.ai Studio** | `/teamspace/studios/this_studio` (persistent) | `rclone` remote |
| **Google Colab** | `/content` (lost when the session ends) | `drive.mount()` |
| **any other Linux machine** | `~/dgs-benchmark` | `rclone` remote |

Detection is automatic; `BENCH_PLATFORM` and `BENCH_WORKDIR` override it if needed.

Cell 0.0 also defines the `env_*` helpers that make **every setting in cell 0.1
overridable from an environment variable**, so the notebook can be driven from a terminal
over SSH without editing a single cell. Each setting keeps its literal as the default, so
with no variables set the notebook behaves exactly as if you had edited cell 0.1 by hand.
How to drive a run from a terminal: see the section *Running on a cloud GPU over SSH* of the repository [README](https://github.com/riccardolanza05/dynamic-gaussian-splatting-benchmark#running-on-a-cloud-gpu-over-ssh).

In [ ]:
# 0.0  Where are we running, and how are settings overridden?
import os
import shutil
import subprocess
import sys


def detect_platform():
    """"colab", "lightning" or "generic". Never raises."""
    if "google.colab" in sys.modules:
        return "colab"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except Exception:
        pass
    # A lightning.ai Studio always has this persistent directory.
    if os.path.isdir("/teamspace/studios/this_studio"):
        return "lightning"
    return "generic"


PLATFORM = os.environ.get("BENCH_PLATFORM") or detect_platform()

DEFAULT_WORKDIR = {
    "colab": "/content",
    "lightning": "/teamspace/studios/this_studio",
}.get(PLATFORM, os.path.expanduser("~/dgs-benchmark"))

# Everything this notebook creates -- cloned repositories, the dataset cache, the models,
# the results -- lives under WORKDIR. On a Studio it is persistent; on Colab it is not.
WORKDIR = os.environ.get("BENCH_WORKDIR") or DEFAULT_WORKDIR
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)


# Settings from the environment: every value in cell 0.1 can be overridden with a
# BENCH_<NAME> variable, which is what makes an unattended SSH run possible. With none
# set, the literals written in cell 0.1 are used.

def env_str(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else value


def env_int(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else int(value)


def env_float(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else float(value)


def env_bool(name, default):
    value = os.environ.get("BENCH_" + name)
    if value is None or value == "":
        return default
    return value.strip().lower() in ("1", "true", "yes", "on")


def env_list(name, default):
    """A comma-separated list, e.g. BENCH_SCENES="sear_steak,flame_steak"."""
    value = os.environ.get("BENCH_" + name)
    if value is None or value == "":
        return list(default)
    return [item.strip() for item in value.split(",") if item.strip()]


# Installing system packages needs root on a Studio but not on Colab.
if os.geteuid() == 0:
    SUDO = ""
elif shutil.which("sudo"):
    SUDO = "sudo -n "
else:
    SUDO = ""

print("Platform    : %s" % PLATFORM)
print("Working dir : %s" % WORKDIR)
_overrides = sorted(k for k in os.environ if k.startswith("BENCH_"))
print("Overrides   : %s" % (", ".join(_overrides) if _overrides else "none (using the "
                            "literals in cell 0.1)"))
try:
    print()
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True).stdout.strip()
          or "nvidia-smi produced no output: is a GPU attached?")
except FileNotFoundError:
    print("nvidia-smi not found: this machine has no NVIDIA driver. Training will fail.")


## 0. User configuration

Everything that changes between runs is set in cell 0.1; no other cell needs editing.

**Storage.** `STORAGE_MODE = "drive"` writes the benchmark JSON (and, with `KEEP_MODEL_ON_DRIVE = True`, the model) to Google Drive, so a run survives a runtime restart and the loop can resume. `"local"` keeps everything in `/content`, which is lost when the session ends.

**What to run.** `RUN_MODE = "single"` trains `SCENE`; `"loop"` trains every scene in `SCENES_TO_RUN`. Completed runs are skipped, so re-running a loop after a disconnection resumes from the first missing scene.

**Which protocol.**

| | Protocol A, `"iterations"` | Protocol B, `"target_eval_loss"` |
|---|---|---|
| Stopping criterion | fixed budget `MAX_ITERATIONS` | per-scene target in `TARGET_EVAL_LOSS_PER_SCENE` |
| Metric sampling | every `EVAL_EVERY_N_ITERS` iterations | every `EVAL_EVERY_N_MINUTES` minutes |
| Question answered | at equal budget, which method reconstructs best? | to reach the same quality, what does each method cost? |

**Workflow.** The Protocol B target must be reachable by every method, so it is calibrated from Protocol A: run the Protocol A loop on all scenes in all four notebooks, read the candidates printed by cell 4.3, take the per-scene maximum across the methods and use the same dictionary in all four notebooks. **The targets ship as `None`**, because no multi-view run exists yet; with `None` the Protocol B loop skips the scene instead of inventing a bar.

**The temporal window.** `NUM_FRAMES = 50` selects frames 0–49 of every scene (1.67 s at 30 fps) for training *and* for the test split. It is the window in which the four methods coincide: Spacetime Gaussians trains N3DV in 50-frame chunks by construction, and Dynamic 3D Gaussians costs a fixed number of optimiser steps *per frame*, so the full 300-frame sequence is out of reach of a free-tier T4. The window is written into every benchmark JSON (`num_frames`), so runs made with different windows can never be mixed by accident.

**Disk.** One N3DV scene is about 1.2 GB of video and, once the frames are extracted, 15–20 GB of PNG at 1352×1014 for 21 cameras × 50 frames. `DELETE_VIDEOS_AFTER_EXTRACT` and `DELETE_FRAMES_AFTER_TRAIN` keep the machine inside its disk budget when looping over several scenes.

In [ ]:
# 0.1  User configuration. Edit the defaults here, or set the matching
# BENCH_<NAME> environment variable to override one without touching the file.

# Where the results go.
#   "drive" : the shared Google Drive folder of the study. On Colab that folder is mounted
#             directly; anywhere else it is a local staging directory kept in step with
#             Drive by rclone (cell 0.2), so an SSH run needs no browser.
#   "local" : stay on this machine, sync nothing.
STORAGE_MODE = env_str("STORAGE_MODE", "drive")

# ONE folder for the whole multi-view study: the four notebooks write four subfolders
# inside it, named exactly as results/analysis/scripts/study.py expects, so the folder can
# be downloaded and analysed as it is. This is what collects every JSON of every method.
BENCH_ROOT_NAME = env_str("ROOT_NAME", "dgs-benchmark-n3dv")
METHOD_DIR_NAME = "dynamic3dgaussians_output"
# Name of the rclone remote pointing at Google Drive (created once with `rclone config`).
RCLONE_REMOTE = env_str("RCLONE_REMOTE", "gdrive")

# "single" trains SCENE only; "loop" trains every scene in SCENES_TO_RUN, in order.
RUN_MODE = env_str("RUN_MODE", "loop")

SCENE = env_str("SCENE", "sear_steak")

# Used only when RUN_MODE == "loop"; trim the list for a partial loop.
SCENES_TO_RUN = env_list("SCENES", ["coffee_martini", "cook_spinach", "cut_roasted_beef",
                                    "flame_salmon_1", "flame_steak", "sear_steak"])
SCENES_NOT_TO_RUN = []

# Skip runs that are already complete (this makes the loop resumable).
SKIP_TRAINING_IF_TRAINED = env_bool("SKIP_TRAINING_IF_TRAINED", True)

# False: the model stays on local disk and only the benchmark JSON is kept
#        (model storage is measured before the local copy is discarded).
# True:  the model is kept next to the JSON, so Part 2 can run later without retraining.
#        On Drive that is gigabytes per run: leave it False unless you need it.
KEEP_MODEL_ON_DRIVE = env_bool("KEEP_MODEL_ON_DRIVE", False)
LOCAL_TRAIN_ROOT = os.path.join(WORKDIR, "train_outputs")

# --- Multi-view protocol (identical in notebooks 04-07) -----------------------------
# Frames 0..NUM_FRAMES-1 of every scene are used for training and for the test split.
NUM_FRAMES = env_int("NUM_FRAMES", 50)
# N3DV holds out the centre camera; the dataset README states cam00 is the test view.
HELD_OUT_CAMERA = env_str("HELD_OUT_CAMERA", "cam00")
# Evaluation resolution: half of the native 2704x2028, the resolution all four official
# configs use for N3DV. Written as a downscale factor of the native video.
IMAGE_DOWNSCALE = env_int("IMAGE_DOWNSCALE", 2)
# ------------------------------------------------------------------------------------

# Where the raw N3DV videos are cached. One scene is about 1.2 GB.
N3DV_RAW_ROOT = env_str("N3DV_RAW_ROOT", os.path.join(WORKDIR, "n3dv"))
# Delete the .mp4 files once the frames have been extracted (they are not needed again).
DELETE_VIDEOS_AFTER_EXTRACT = env_bool("DELETE_VIDEOS_AFTER_EXTRACT", True)
# Delete the extracted frames of a scene once its run is finished (frees ~15-20 GB).
DELETE_FRAMES_AFTER_TRAIN = env_bool("DELETE_FRAMES_AFTER_TRAIN", True)

# Benchmarking protocol: "iterations" (Protocol A) or "target_eval_loss" (Protocol B).
TRAINING_MODE = env_str("TRAINING_MODE", "iterations")

# Protocol A: fixed iteration budget
MAX_ITERATIONS     = 30000   # recomputed below from the per-timestep budget
EVAL_EVERY_N_ITERS = env_int("EVAL_EVERY_N_ITERS", 1000)  # sample every N iterations

# Protocol B: fixed quality target
# "eval_loss" stops on the per-scene L1 target below, "psnr" on TARGET_PSNR.
TARGET_METRIC = "eval_loss"
TARGET_PSNR   = 30.0    # used only when TARGET_METRIC == "psnr"

# Per-scene L1 target, identical in the four notebooks, to be calibrated from the Protocol A
# results of all four methods (cell 4.3, docs/PROTOCOL_B_CALIBRATION.md).
# A scene left at None is skipped by the Protocol B loop: no multi-view Protocol A run exists
# yet, so every target is None and must be filled in before Protocol B can be run.
TARGET_EVAL_LOSS_PER_SCENE = {
    "coffee_martini":   None,
    "cook_spinach":     None,
    "cut_roasted_beef": None,
    "flame_salmon_1":   None,
    "flame_steak":      None,
    "sear_steak":       None,
}

EVAL_EVERY_N_MINUTES  = 0.5     # sample the metrics every T minutes
SAFETY_MAX_ITERATIONS = 60000   # hard cap, so a run can never last forever
# The target is ignored below this iteration, so the baseline sample cannot stop the run.
MIN_ITERATIONS_BEFORE_STOP = 1000
# The target must hold for this many consecutive samples (test metrics fluctuate).
TARGET_CONSECUTIVE_HITS = 2

# Evaluation settings (shared by both protocols)
# "l1": mean L1 on the test split; "l1_dssim": (1-w)*L1 + w*(1-SSIM). Both are always recorded;
# this selects the one mirrored by "eval_loss" and used by the Protocol B stop.
EVAL_LOSS_KIND = "l1"
LAMBDA_DSSIM   = 0.2      # weight w of the "l1_dssim" variant

LPIPS_NET      = "vgg"    # LPIPS backbone, as reported in all four papers
# One LPIPS backend for the whole multi-view study. Dynamic 3D Gaussians does not bundle
# lpipsPyTorch, and mixing the two backends would put a systematic offset on the LPIPS column,
# so the pip package (with the [-1,1] rescaling it expects) is used by all four notebooks.
LPIPS_BACKEND  = "lpips-pip"
MAX_EVAL_VIEWS = 0        # 0 = every test view (50 frames of the held-out camera)
EVAL_AT_FIRST_ITERATION = True   # record a baseline point at the first iteration

# Extra flags appended to the training command (usually empty)
EXTRA_TRAIN_ARGS = env_str("EXTRA_TRAIN_ARGS", "")

# Views consumed per optimiser step; the official per-scene config may overwrite it.
BATCH_SIZE = 1

# --- Iteration budget of a frame-by-frame method ------------------------------------
# This method does not have a single optimisation over the sequence: it spends a fixed
# number of steps on the first frame and a smaller fixed number on each later frame, so
# its total budget is a function of the window and cannot be set to 30 000 without
# changing the per-frame budget the authors chose.
#
#   "native"  : the authors' schedule, 10 000 + 2 000 per later frame.
#   "aligned" : the total is forced to ALIGNED_TOTAL_ITERATIONS, keeping the first frame
#               at D3DG_ITERS_FIRST and dividing the rest equally over the later frames.
#
# "native" is the default, for the same reason the monocular study did not equalise the
# per-scene batch size of 4DGS native-4D: forcing the unit of work would report a number
# about a method that nobody runs. The choice is recorded in the benchmark JSON, and
# `images_seen` (steps x batch) remains the budget axis the analysis compares on.
D3DG_BUDGET_MODE = env_str("D3DG_BUDGET_MODE", "native")
D3DG_ITERS_FIRST = env_int("D3DG_ITERS_FIRST", 10000)
D3DG_ITERS_PER_TIMESTEP = env_int("D3DG_ITERS_PER_TIMESTEP", 2000)
ALIGNED_TOTAL_ITERATIONS = env_int("ALIGNED_TOTAL_ITERATIONS", 30000)

assert D3DG_BUDGET_MODE in ("native", "aligned")
# This method is not part of the full-length (300-frame) study: its cost is per frame, so
# 300 frames would be 608 000 steps per scene at the native budget. See
# docs/OPEN_DECISIONS_MULTIVIEW.md, decision 2, option D.
D3DG_MAX_FRAMES = 50
assert NUM_FRAMES <= D3DG_MAX_FRAMES, (
    "Dynamic 3D Gaussians is run on at most %d frames (NUM_FRAMES = %d); the "
    "full-length study covers notebooks 05, 06 and 07 only." % (D3DG_MAX_FRAMES, NUM_FRAMES))
if D3DG_BUDGET_MODE == "aligned":
    D3DG_ITERS_PER_TIMESTEP = max(
        1, (ALIGNED_TOTAL_ITERATIONS - D3DG_ITERS_FIRST) // max(1, NUM_FRAMES - 1))

MAX_ITERATIONS = D3DG_ITERS_FIRST + D3DG_ITERS_PER_TIMESTEP * (NUM_FRAMES - 1)
SAFETY_MAX_ITERATIONS = MAX_ITERATIONS
# The natural sampling grid of a frame-by-frame method is the frame: sampling every
# D3DG_ITERS_PER_TIMESTEP steps puts one sample at the end of every frame, plus a few
# inside the longer first frame.
EVAL_EVERY_N_ITERS = D3DG_ITERS_PER_TIMESTEP
print("Budget   : %s, %d + %d x %d = %d steps"
      % (D3DG_BUDGET_MODE, D3DG_ITERS_FIRST, D3DG_ITERS_PER_TIMESTEP,
         NUM_FRAMES - 1, MAX_ITERATIONS))
# ------------------------------------------------------------------------------------

# Dynamic/static segmentation. N3DV ships no masks, and this method needs one per image
# (its `seg` loss and its foreground/background split). The masks are derived from the
# temporal median of each camera; see cell 3.5.
SEG_DIFF_THRESHOLD = 0.08   # mean absolute difference from the median, in [0, 1]
SEG_MIN_AREA_RATIO = 0.0005  # drop connected components smaller than this fraction of the frame

METHOD_NAME = "Dynamic 3D Gaussians (Luiten et al.)"
REPO_DIR = os.path.join(WORKDIR, "Dynamic3DGaussians")
N3DV_SCENES = ["coffee_martini", "cook_spinach", "cut_roasted_beef",
               "flame_salmon_1", "flame_steak", "sear_steak"]

assert STORAGE_MODE in ("drive", "local")
assert RUN_MODE in ("single", "loop")
assert TRAINING_MODE in ("iterations", "target_eval_loss")
assert EVAL_LOSS_KIND in ("l1", "l1_dssim")
assert TARGET_METRIC in ("psnr", "eval_loss")
assert LPIPS_BACKEND in ("lpips-pip", "lpipsPyTorch", "auto")
assert STORAGE_MODE in ("drive", "local")
assert 1 <= NUM_FRAMES <= 300, "N3DV provides 300 frames per camera"
assert SCENE in N3DV_SCENES, "Unknown scene: %s" % SCENE
assert all(s in N3DV_SCENES for s in SCENES_TO_RUN)
print("Configuration accepted.")
print("Method   : %s" % METHOD_NAME)
print("Run mode : %s" % RUN_MODE)
print("Protocol : %s" % TRAINING_MODE)
print("Window   : frames 0..%d, test camera %s" % (NUM_FRAMES - 1, HELD_OUT_CAMERA))
print("Scenes   : %s" % (SCENE if RUN_MODE == "single" else ", ".join(SCENES_TO_RUN)))

In [ ]:
# 0.2  Where the results live, and how they reach Google Drive.
import glob
import json
import os
import shutil
import subprocess

# STUDY_ROOT is the one folder that holds all four methods; OUTPUT_ROOT is this method's
# subfolder of it. The layout is the one the analysis pipeline reads directly:
#
#   <STUDY_ROOT>/
#       dynamic3dgaussians_output/   <run>/benchmark/benchmark_*.json      (notebook 04)
#       4dgaussian_n3dv_output/      ...                                   (notebook 05)
#       4dgs_fudan_n3dv_output/      ...                                   (notebook 06)
#       spacetime_gaussians_output/  ...                                   (notebook 07)
#
DRIVE_SYNC = None          # rclone destination, or None when nothing is synced

if STORAGE_MODE == "drive" and PLATFORM == "colab":
    # Colab can mount Drive as a filesystem: write straight into it.
    from google.colab import drive
    drive.mount("/content/drive")
    STUDY_ROOT = os.path.join("/content/drive/MyDrive", BENCH_ROOT_NAME)
elif STORAGE_MODE == "drive":
    # Everywhere else (a lightning.ai Studio, any Linux box): write locally and keep the
    # folder in step with Drive through rclone, which needs no browser once configured.
    STUDY_ROOT = os.path.join(WORKDIR, "bench_out", BENCH_ROOT_NAME)
    DRIVE_SYNC = "%s:%s" % (RCLONE_REMOTE, BENCH_ROOT_NAME)
else:
    STUDY_ROOT = os.path.join(WORKDIR, "bench_out", BENCH_ROOT_NAME)

OUTPUT_ROOT = os.path.join(STUDY_ROOT, METHOD_DIR_NAME)

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOCAL_TRAIN_ROOT, exist_ok=True)
os.makedirs(N3DV_RAW_ROOT, exist_ok=True)
TRAINED_GLOB = ("params.npz",)


def _rclone(*args):
    """Run rclone, returning True on success. Never raises: a failed sync must not
    destroy a finished training run, it must only be reported."""
    if shutil.which("rclone") is None:
        print("[sync] rclone is not installed; results stay in %s only." % OUTPUT_ROOT)
        return False
    command = ["rclone"] + list(args)
    print("[sync] $ %s" % " ".join(command), flush=True)
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print("[sync] FAILED (%d): %s" % (result.returncode,
                                          (result.stderr or "").strip()[-400:]))
        return False
    return True


def sync_pull():
    """Fetch this method's results back from Drive, so a fresh machine can resume.

    Only the benchmark folders are pulled, never the models: what makes the loop
    resumable is the JSON, and the models can be gigabytes.
    """
    if not DRIVE_SYNC:
        return
    # */blocks/** holds the per-block results of a run trained in blocks (notebook 07 at
    # more than 50 frames), so a half-finished scene resumes at the first missing block.
    _rclone("copy", "%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME), OUTPUT_ROOT,
            "--include", "*/benchmark/**", "--include", "*/blocks/**",
            "--create-empty-src-dirs")


def sync_push(run=None):
    """Send results to Drive. With a run name, only that run; otherwise everything."""
    if not DRIVE_SYNC:
        return
    if run:
        source = os.path.join(OUTPUT_ROOT, run)
        destination = "%s/%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME, run)
    else:
        source, destination = OUTPUT_ROOT, "%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME)
    if not os.path.isdir(source):
        return
    # copy, not sync: nothing on Drive is ever deleted by a run.
    if _rclone("copy", source, destination, "--transfers", "4"):
        print("[sync] %s -> %s" % (source, destination))


def run_name(scene, mode):
    """Folder name for one (scene, protocol) pair. Runs never collide.

    The frame window is part of the name: multi-view runs made with different
    windows are different experiments and must not share a folder.
    """
    # RUN_TAG names a model variant the rest of the name does not capture (notebook 07 sets
    # it for ours_full), so two variants never share a folder or reuse each other's JSON.
    suffix = globals().get("RUN_TAG", "")
    if mode == "iterations":
        return "%s_f%d_iters%d%s" % (scene, NUM_FRAMES, MAX_ITERATIONS, suffix)
    target = TARGET_EVAL_LOSS_PER_SCENE.get(scene)
    tag = ("%g" % target).replace(".", "p") if target is not None else "NA"
    return "%s_f%d_loss%s%s" % (scene, NUM_FRAMES, tag, suffix)


def raw_scene_dir(scene):
    """Where the raw N3DV videos and poses_bounds.npy of one scene live."""
    return os.path.join(N3DV_RAW_ROOT, scene)


def source_path_for(scene):
    """Where this notebook writes the Panoptic-style view of the scene (train_meta.json, test_meta.json, ims/, seg/ and init_pt_cld.npz)."""
    return os.path.join(REPO_DIR, "data", scene)


def paths_for(scene, mode):
    """All output paths for one run.

    The benchmark JSON always lives on OUTPUT_ROOT (Drive when selected), while
    the model goes to Drive only if KEEP_MODEL_ON_DRIVE, otherwise to local
    disk. Metrics are what the benchmark needs; the checkpoints are optional.
    """
    name = run_name(scene, mode)
    model_root = OUTPUT_ROOT if KEEP_MODEL_ON_DRIVE else LOCAL_TRAIN_ROOT
    model_path = os.path.join(model_root, name)
    bench_dir = os.path.join(OUTPUT_ROOT, name, "benchmark")
    return {
        "scene": scene,
        "mode": mode,
        "run_name": name,
        "model_path": model_path,
        "drive_model_path": os.path.join(OUTPUT_ROOT, name),
        "benchmark_dir": bench_dir,
        "bench_json": os.path.join(bench_dir, "benchmark_%s.json" % mode),
        "bench_config": os.path.join(bench_dir, "benchmark_config.json"),
        "source_path": source_path_for(scene),
        "raw_path": raw_scene_dir(scene),
    }


def is_model_trained(model_path):
    """True when the model files of that run are present on disk."""
    return len(glob.glob(os.path.join(model_path, *TRAINED_GLOB))) > 0


TERMINAL_STATUS = ("finished", "stopped_on_target", "stopped_on_safety_cap")


def run_is_complete(p):
    """True when this run has already produced a finished benchmark JSON.

    Completion is judged on the JSON rather than on the model, because with
    KEEP_MODEL_ON_DRIVE = False the model does not survive the runtime. This is
    what makes the loop resumable after a disconnection or a GPU-quota stop.
    """
    if not os.path.exists(p["bench_json"]):
        return False
    try:
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
    except Exception:
        return False
    if summary.get("status") not in TERMINAL_STATUS:
        return False
    if summary.get("num_frames") not in (None, NUM_FRAMES):
        return False
    if KEEP_MODEL_ON_DRIVE and not is_model_trained(p["drive_model_path"]):
        return False
    return True


def activate_scene(scene, mode=None):
    """Point the module-level variables at one scene. Part 2 reads these."""
    global SCENE, SOURCE_PATH, MODEL_OUTPUT_PATH, MODEL_DIR, RAW_PATH
    global BENCHMARK_DIR, BENCH_JSON_PATH, BENCH_CONFIG_PATH, RUN_NAME
    mode = mode or TRAINING_MODE
    p = paths_for(scene, mode)
    SCENE = scene
    SOURCE_PATH = p["source_path"]
    RAW_PATH = p["raw_path"]
    MODEL_OUTPUT_PATH = p["model_path"]
    MODEL_DIR = p["model_path"]          # alias used by Part 2
    BENCHMARK_DIR = p["benchmark_dir"]
    BENCH_JSON_PATH = p["bench_json"]
    BENCH_CONFIG_PATH = p["bench_config"]
    RUN_NAME = run_name(scene, mode)
    os.makedirs(BENCHMARK_DIR, exist_ok=True)
    return p


activate_scene(SCENE)

# A fresh Studio starts empty: bring back what previous runs already produced, so the
# loop skips them instead of retraining them.
sync_pull()

print("Study root   : %s" % STUDY_ROOT)
print("Metrics root : %s" % OUTPUT_ROOT)
print("Drive sync   : %s" % (DRIVE_SYNC or "off (results stay on this machine)"))
print("Model folder : %s%s" % (MODEL_OUTPUT_PATH,
                               "" if KEEP_MODEL_ON_DRIVE else "   (not kept)"))
print("Benchmark    : %s" % BENCH_JSON_PATH)
print("Already done : %s" % run_is_complete(paths_for(SCENE, TRAINING_MODE)))

## 1. Environment and repository setup

In [ ]:
# The GPU this run will use (cell 0.0 already printed a summary)
!nvidia-smi

In [ ]:
import os
os.chdir(WORKDIR)
!git clone https://github.com/JonathonLuiten/Dynamic3DGaussians
%cd Dynamic3DGaussians

# The rasterizer is the author's own fork (it also returns depth)
!git clone https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth

In [ ]:
# Reuse the PyTorch/CUDA already installed and add the remaining dependencies.
# open3d is in environment.yml but has no wheel for every Python version;
# it is only used for a k-NN query and for the standalone visualiser (see cell 2).
!pip install -q tqdm Pillow plyfile lpips opencv-python-headless scipy "imageio[ffmpeg]" matplotlib

In [ ]:
import os

!pip install -q "setuptools<70" wheel ninja
%env TORCH_CUDA_ARCH_LIST=7.5

os.chdir(os.path.join(REPO_DIR, 'diff-gaussian-rasterization-w-depth'))
!pip install . --no-build-isolation
os.chdir(REPO_DIR)
print('Rasterizer installed.')

## 2. Environment fixes

* **`o3d_knn`.** `helpers.py` uses Open3D's KD-tree for one query: the 3 nearest neighbours of every initial point, whose mean distance sets the initial Gaussian scale, and the 20 nearest neighbours of the foreground points, which define the rigidity neighbourhood. Open3D has no wheel for every Python version. The function is replaced by a `scipy.spatial.cKDTree` implementation that returns the same two arrays — squared distances and indices of the `k` nearest neighbours, excluding the point itself — so the numerical behaviour is unchanged. A pristine copy is kept as `helpers.py.orig`.
* **Iteration budget.** `train.py` hard-codes `10000 if is_initial_timestep else 2000`. That line is made to read `BENCH_D3DG_ITERS_FIRST` and `BENCH_D3DG_ITERS_PER_TS`, defaulting to the upstream values when the variables are unset, so that the budget knobs of cell 0.1 reach the loop. This is the only change to training-loop code in the four notebooks and it changes nothing without the environment variables.
* **Entry point.** `train.py`'s `__main__` block hard-codes the six Panoptic Sports sequences. A separate `train_one.py` is written next to it, which imports `train` and calls it for one sequence; `train.py` itself is not touched apart from the monitor hook.

In [ ]:
# Replace Open3D's KD-tree with scipy's; same inputs, same two outputs.
import os
import shutil

target = os.path.join(REPO_DIR, "helpers.py")
backup = target + ".orig"
if not os.path.exists(backup):
    shutil.copyfile(target, backup)

with open(backup) as handle:
    source = handle.read()

OLD_IMPORT = "import open3d as o3d\n"
NEW_IMPORT = ("# open3d has no wheel for this Python version; o3d_knn below uses scipy instead.\n"
              "from scipy.spatial import cKDTree\n")
assert source.count(OLD_IMPORT) == 1, "unexpected helpers.py layout"

OLD_KNN_START = "def o3d_knn(pts, num_knn):"
start = source.index(OLD_KNN_START)
end = source.index("def params2cpu(params, is_initial_timestep):")
NEW_KNN = '''def o3d_knn(pts, num_knn):
    # k nearest neighbours of every point, excluding the point itself.
    # Drop-in replacement for the Open3D KD-tree version: returns the same
    # (squared distances, indices) pair, in the same order.
    tree = cKDTree(np.ascontiguousarray(pts, np.float64))
    dists, indices = tree.query(np.ascontiguousarray(pts, np.float64), k=num_knn + 1)
    return np.ascontiguousarray(dists[:, 1:] ** 2), np.ascontiguousarray(indices[:, 1:])


'''
patched = source[:start] + NEW_KNN + source[end:]
patched = patched.replace(OLD_IMPORT, NEW_IMPORT, 1)

with open(target, "w") as handle:
    handle.write(patched)
print("helpers.py: o3d_knn now uses scipy.spatial.cKDTree (pristine copy in helpers.py.orig).")

In [ ]:
# Make the per-timestep iteration budget configurable.
# Upstream: num_iter_per_timestep = 10000 if is_initial_timestep else 2000
# Patched:  the same expression, reading two environment variables that default to those
# values, so without them train.py behaves exactly as upstream.
import os
import shutil

target = os.path.join(REPO_DIR, "train.py")
backup = target + ".budget_orig"
if not os.path.exists(backup):
    shutil.copyfile(target, backup)

with open(backup) as handle:
    source = handle.read()

OLD = "        num_iter_per_timestep = 10000 if is_initial_timestep else 2000\n"
NEW = ("        num_iter_per_timestep = (int(os.environ.get('BENCH_D3DG_ITERS_FIRST', 10000))\n"
       "                                 if is_initial_timestep else\n"
       "                                 int(os.environ.get('BENCH_D3DG_ITERS_PER_TS', 2000)))\n")
assert source.count(OLD) == 1, "unexpected train.py layout"

with open(target, "w") as handle:
    handle.write(source.replace(OLD, NEW, 1))
print("train.py: the per-timestep budget now reads BENCH_D3DG_ITERS_FIRST / _PER_TS.")

# train.py already imports os at module level; verify rather than assume.
with open(target) as handle:
    assert "\nimport os\n" in handle.read(), "os is not imported in train.py"
print("Verified: os is imported in train.py.")

In [ ]:
# A single-sequence entry point. train.py's __main__ block hard-codes the six Panoptic
# Sports sequences; this launcher imports train() and runs it for one sequence, then moves
# the output where the benchmark driver expects it. train.py itself is untouched.
import os

LAUNCHER = '''
# train_one.py -- run Dynamic3DGaussians on one sequence (written by notebook 04).
import os
import shutil
import sys

from train import train

if __name__ == "__main__":
    seq = sys.argv[1]
    exp = sys.argv[2]
    destination = os.environ.get("BENCH_D3DG_MODEL_PATH", "")
    produced = os.path.join("./output", exp, seq)
    shutil.rmtree(produced, ignore_errors=True)
    try:
        train(seq, exp)
    finally:
        # Runs on the normal path and on the monitor's sys.exit(0) in Protocol B.
        if destination and os.path.isdir(produced):
            shutil.rmtree(destination, ignore_errors=True)
            parent = os.path.dirname(destination)
            if parent:
                os.makedirs(parent, exist_ok=True)
            shutil.move(produced, destination)
            print("[launcher] model moved to %s" % destination, flush=True)
    print("Training complete.")
'''

with open(os.path.join(REPO_DIR, "train_one.py"), "w") as handle:
    handle.write(LAUNCHER)
print("train_one.py written to %s" % os.path.join(REPO_DIR, "train_one.py"))

## 3. N3DV dataset download

The *Neural 3D Video Synthesis from Multi-View Video* dataset is downloaded **one scene at a time**, because a single scene is about 1.2 GB of video and 15–20 GB of extracted frames, well beyond what a single machine can comfortably hold for six scenes at once. The download is cached under `N3DV_RAW_ROOT` and skipped when the scene is already there, so the cell is safe to re-run.

Each scene ships as `cam00.mp4 … cam20.mp4` plus `poses_bounds.npy` in the LLFF pose format. `cam00` is the centre reference camera and is **held out for testing** — that is the dataset's own convention, stated in its README, and the convention all four methods follow. `flame_salmon_1` is published as a split archive and is rejoined before extraction.

Frames are extracted at **1352×1014** (`IMAGE_DOWNSCALE = 2`), the resolution used by the official N3DV configuration of every method in this comparison.

In [ ]:
# 3  N3DV download and frame extraction, one scene at a time (resumable).
import os
import shutil
import subprocess

N3DV_RELEASE = ("https://github.com/facebookresearch/Neural_3D_Video/"
                "releases/download/v1.0")
# flame_salmon_1 is published as a 4-part split zip; the others as a single archive.
N3DV_SPLIT_SCENES = {"flame_salmon_1": ["flame_salmon_1_split.z01",
                                        "flame_salmon_1_split.z02",
                                        "flame_salmon_1_split.z03",
                                        "flame_salmon_1_split.zip"]}


def _sh(cmd, cwd=None):
    print("$ %s" % cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    assert proc.returncode == 0, "command failed (%d): %s" % (proc.returncode, cmd)


def scene_is_downloaded(scene):
    d = raw_scene_dir(scene)
    if not os.path.exists(os.path.join(d, "poses_bounds.npy")):
        return False
    return len([f for f in os.listdir(d) if f.endswith(".mp4")]) > 0


def scene_is_extracted(scene, num_frames=None):
    """True when every camera already has the frames of the window on disk."""
    num_frames = NUM_FRAMES if num_frames is None else num_frames
    d = raw_scene_dir(scene)
    cams = sorted(c for c in os.listdir(d)
                  if os.path.isdir(os.path.join(d, c)) and c.startswith("cam"))
    if not cams:
        return False
    for cam in cams:
        frames = os.path.join(d, cam, "frames")
        if not os.path.isdir(frames):
            return False
        if any(not os.path.exists(os.path.join(frames, "%04d.png" % i))
               for i in range(num_frames)):
            return False
    return True


def download_scene(scene):
    """Fetch one N3DV scene into raw_scene_dir(scene). Skipped when present."""
    target = raw_scene_dir(scene)
    if scene_is_downloaded(scene):
        print("[%s] videos already present, download skipped." % scene)
        return target
    os.makedirs(target, exist_ok=True)
    tmp = os.path.join(N3DV_RAW_ROOT, "_dl_%s" % scene)
    os.makedirs(tmp, exist_ok=True)
    if scene in N3DV_SPLIT_SCENES:
        for part in N3DV_SPLIT_SCENES[scene]:
            dst = os.path.join(tmp, part)
            if not os.path.exists(dst):
                _sh('wget -q --show-progress -O "%s" "%s/%s"' % (dst, N3DV_RELEASE, part))
        joined = os.path.join(tmp, "joined.zip")
        if not os.path.exists(joined):
            _sh('zip -q -s 0 "%s/%s" --out "%s"'
                % (tmp, N3DV_SPLIT_SCENES[scene][-1], joined))
        _sh('unzip -q -o "%s" -d "%s"' % (joined, tmp))
    else:
        archive = os.path.join(tmp, "%s.zip" % scene)
        if not os.path.exists(archive):
            _sh('wget -q --show-progress -O "%s" "%s/%s.zip"'
                % (archive, N3DV_RELEASE, scene))
        _sh('unzip -q -o "%s" -d "%s"' % (archive, tmp))

    # The archives unpack either flat or inside a <scene>/ folder; normalise both.
    src = tmp
    if not os.path.exists(os.path.join(src, "poses_bounds.npy")):
        candidates = [os.path.join(src, d) for d in os.listdir(src)
                      if os.path.isdir(os.path.join(src, d))
                      and os.path.exists(os.path.join(src, d, "poses_bounds.npy"))]
        assert candidates, "poses_bounds.npy not found in the archive of %s" % scene
        src = candidates[0]
    for name in os.listdir(src):
        if name.endswith(".mp4") or name == "poses_bounds.npy":
            shutil.move(os.path.join(src, name), os.path.join(target, name))
    shutil.rmtree(tmp, ignore_errors=True)
    print("[%s] downloaded to %s" % (scene, target))
    return target


def extract_frames(scene):
    """Decode the first NUM_FRAMES frames of every camera at 1352x1014.

    Frames land in <raw>/<camNN>/frames/%04d.png, which is the layout every
    per-method converter below reads. Cameras keep the names of their videos, so
    the held-out camera stays identifiable whatever the pose ordering is.
    """
    d = raw_scene_dir(scene)
    videos = sorted(f for f in os.listdir(d) if f.endswith(".mp4"))
    assert videos, "no videos in %s; run download_scene first" % d
    for video in videos:
        cam = os.path.splitext(video)[0]
        out = os.path.join(d, cam, "frames")
        os.makedirs(out, exist_ok=True)
        done = all(os.path.exists(os.path.join(out, "%04d.png" % i))
                   for i in range(NUM_FRAMES))
        if done:
            continue
        scale = ("-vf scale=iw/%d:ih/%d" % (IMAGE_DOWNSCALE, IMAGE_DOWNSCALE)
                 if IMAGE_DOWNSCALE != 1 else "")
        _sh('ffmpeg -y -loglevel error -i "%s" -vframes %d %s -start_number 0 "%s/%%04d.png"'
            % (os.path.join(d, video), NUM_FRAMES, scale, out))
    if DELETE_VIDEOS_AFTER_EXTRACT:
        for video in videos:
            os.remove(os.path.join(d, video))
        print("[%s] videos removed after extraction (poses_bounds.npy kept)." % scene)
    print("[%s] %d frames per camera at downscale %d."
          % (scene, NUM_FRAMES, IMAGE_DOWNSCALE))


def ensure_scene_raw(scene):
    """Download and extract one scene if needed; returns its raw folder."""
    if not scene_is_extracted(scene):
        if not scene_is_downloaded(scene):
            download_scene(scene)
        extract_frames(scene)
    else:
        print("[%s] frames already extracted, nothing to do." % scene)
    return raw_scene_dir(scene)


def drop_scene_frames(scene):
    """Free the extracted frames of a scene once its run is finished."""
    d = raw_scene_dir(scene)
    for cam in sorted(os.listdir(d)):
        frames = os.path.join(d, cam, "frames")
        if os.path.isdir(frames):
            shutil.rmtree(frames, ignore_errors=True)
    print("[%s] extracted frames discarded." % scene)


def n3dv_cameras(scene):
    """Camera names of a scene, in the order of poses_bounds.npy.

    The dataset README warns that unsynchronised streams were removed, so the
    numbering has gaps and poses_bounds.npy follows the *sorted file order*, not
    the camera index. Sorting the names reproduces that order exactly.
    """
    d = raw_scene_dir(scene)
    cams = sorted(c for c in os.listdir(d)
                  if os.path.isdir(os.path.join(d, c)) and c.startswith("cam"))
    if not cams:
        cams = sorted(os.path.splitext(f)[0] for f in os.listdir(d)
                      if f.endswith(".mp4"))
    return cams


print("Dataset helpers ready. Raw root: %s" % N3DV_RAW_ROOT)
print("Scenes already extracted: %s"
      % [s for s in N3DV_SCENES
         if os.path.isdir(raw_scene_dir(s)) and scene_is_extracted(s)])

### 3.4 Initial point cloud (shared COLMAP stage)

Three of the four methods start from a COLMAP reconstruction of the **first frame** of the rig — 21 images, known poses — and differ only in how they post-process it: 4DGaussians voxel-downsamples it to at most 40 000 points, 4DGS native-4D randomly subsamples it to `num_pts`, Dynamic 3D Gaussians uses it as it is. Notebooks `04`, `05` and `06` therefore run **the same stage with the same settings**, so the initialisation is not a hidden difference between the methods. (Spacetime Gaussians is the exception: it builds one COLMAP model per frame, which is part of its own 4D initialisation, so notebook `07` has its own stage.)

The poses are not estimated: they come from `poses_bounds.npy` and are written into the COLMAP model, so only triangulation and, when available, dense fusion are run. `COLMAP_DENSE = True` asks for `patch_match_stereo` + `stereo_fusion`; where the machine's COLMAP is built without CUDA that step cannot run, and the cell falls back to the sparse triangulated points and records the choice in `init_point_cloud`, which is written into every benchmark JSON.

In [ ]:
# 3.4  Shared COLMAP stage: one initial point cloud per scene, identical in notebooks 04-06.
import os
import shutil
import subprocess

import numpy as np

# Dense fusion gives a much denser cloud but needs a CUDA-enabled COLMAP; when it is not
# available the sparse triangulated points are used instead and the fallback is recorded.
COLMAP_DENSE = True
COLMAP_KIND = None   # filled in by ensure_point_cloud(); goes into the benchmark JSON


def read_ply(path):
    """Points and 0-255 colours of a PLY, whatever COLMAP wrote."""
    from plyfile import PlyData

    data = PlyData.read(path)["vertex"]
    xyz = np.vstack([data["x"], data["y"], data["z"]]).T.astype(np.float64)
    if "red" in data.data.dtype.names:
        rgb = np.vstack([data["red"], data["green"], data["blue"]]).T.astype(np.float64)
        if rgb.max() <= 1.0 + 1e-6:
            rgb = rgb * 255.0
    else:
        rgb = np.full_like(xyz, 128.0)
    return xyz, rgb


def write_ply(path, xyz, rgb):
    """Canonical point-cloud file: x y z nx ny nz red green blue.

    COLMAP's sparse export has no normals, and two of the three readers that
    consume this file require them, so every cloud is re-emitted in one format.
    """
    import numpy as _np
    from plyfile import PlyData, PlyElement

    dtype = [("x", "f4"), ("y", "f4"), ("z", "f4"),
             ("nx", "f4"), ("ny", "f4"), ("nz", "f4"),
             ("red", "u1"), ("green", "u1"), ("blue", "u1")]
    elements = _np.empty(xyz.shape[0], dtype=dtype)
    elements["x"], elements["y"], elements["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    elements["nx"] = elements["ny"] = elements["nz"] = 0.0
    clipped = _np.clip(rgb, 0, 255).astype(_np.uint8)
    elements["red"], elements["green"], elements["blue"] = (
        clipped[:, 0], clipped[:, 1], clipped[:, 2])
    PlyData([PlyElement.describe(elements, "vertex")]).write(path)


def voxel_downsample(xyz, rgb, max_points=40000, voxel_size=0.02, step=0.01):
    """The voxel downsampling of 4DGaussians' scripts/downsample_point.py.

    That script calls open3d, which has no wheel for every Python version
    currently ships; this is the same algorithm (average the points of each
    voxel, grow the voxel until the cloud is small enough) in numpy.
    """
    while len(xyz) > max_points:
        keys = np.floor(xyz / voxel_size).astype(np.int64)
        _, index, inverse = np.unique(keys, axis=0, return_index=True,
                                      return_inverse=True)
        counts = np.bincount(inverse, minlength=len(index)).astype(np.float64)
        new_xyz = np.zeros((len(index), 3))
        new_rgb = np.zeros((len(index), 3))
        for c in range(3):
            new_xyz[:, c] = np.bincount(inverse, weights=xyz[:, c],
                                        minlength=len(index)) / counts
            new_rgb[:, c] = np.bincount(inverse, weights=rgb[:, c],
                                        minlength=len(index)) / counts
        xyz, rgb = new_xyz, new_rgb
        print("Downsampled points: %d (voxel %.3f)" % (len(xyz), voxel_size))
        voxel_size += step
    return xyz, rgb


def _have_colmap():
    return shutil.which("colmap") is not None


def install_colmap():
    if _have_colmap():
        print("colmap already installed: %s" % shutil.which("colmap"))
        return
    # SUDO is "" on Colab (already root) and "sudo -n " on a Studio.
    _sh(SUDO + "apt-get -qq update")
    _sh(SUDO + "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap")
    print("colmap installed: %s" % shutil.which("colmap"))


def _rotmat2qvec(R):
    """COLMAP quaternion (w, x, y, z) of a rotation matrix; the algorithm COLMAP itself uses."""
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec


def llff_poses(scene):
    """Camera-to-world matrices, focal length and image size from poses_bounds.npy.

    poses_bounds.npy is the LLFF format: per camera a 3x5 block whose last column
    holds (H, W, focal) of the *native* video, plus the two depth bounds. The
    rotation columns are reordered from LLFF's [down, right, backwards] to the
    OpenCV convention [right, down, forwards] that COLMAP expects.
    """
    arr = np.load(os.path.join(raw_scene_dir(scene), "poses_bounds.npy"))
    poses = arr[:, :-2].reshape([-1, 3, 5])
    bounds = arr[:, -2:]
    hwf = poses[0, :, -1]
    height, width, focal = float(hwf[0]), float(hwf[1]), float(hwf[2])
    # LLFF -> OpenCV: [down, right, back] -> [right, down, forward]
    c2w = np.concatenate([poses[:, :, 1:2], -poses[:, :, 0:1],
                          -poses[:, :, 2:3], poses[:, :, 3:4]], axis=2)
    bottom = np.tile(np.array([0.0, 0.0, 0.0, 1.0]).reshape(1, 1, 4), (c2w.shape[0], 1, 1))
    c2w = np.concatenate([c2w, bottom], axis=1)
    scale = float(IMAGE_DOWNSCALE)
    return c2w, focal / scale, width / scale, height / scale, bounds


def _write_colmap_text_model(scene, workdir):
    """Write cameras.txt / images.txt / points3D.txt for the known poses of frame 0."""
    c2w, focal, width, height, _bounds = llff_poses(scene)
    cams = n3dv_cameras(scene)
    assert len(cams) == c2w.shape[0], (
        "poses_bounds.npy has %d poses but %d cameras were found"
        % (c2w.shape[0], len(cams)))
    sparse = os.path.join(workdir, "created", "sparse")
    images_dir = os.path.join(workdir, "images")
    os.makedirs(sparse, exist_ok=True)
    os.makedirs(images_dir, exist_ok=True)

    with open(os.path.join(sparse, "cameras.txt"), "w") as handle:
        handle.write("1 PINHOLE %d %d %f %f %f %f\n"
                     % (int(width), int(height), focal, focal,
                        int(width) // 2, int(height) // 2))

    with open(os.path.join(sparse, "images.txt"), "w") as handle:
        for idx, cam in enumerate(cams, start=1):
            src = os.path.join(raw_scene_dir(scene), cam, "frames", "0000.png")
            dst = os.path.join(images_dir, "%s.png" % cam)
            if not os.path.exists(dst):
                os.symlink(os.path.abspath(src), dst)
            w2c = np.linalg.inv(c2w[idx - 1])
            qvec = _rotmat2qvec(w2c[:3, :3])
            tvec = w2c[:3, 3]
            handle.write("%d %f %f %f %f %f %f %f 1 %s.png\n\n"
                         % (idx, qvec[0], qvec[1], qvec[2], qvec[3],
                            tvec[0], tvec[1], tvec[2], cam))

    open(os.path.join(sparse, "points3D.txt"), "w").close()
    return sparse, images_dir


def ensure_point_cloud(scene):
    """Build (once) the frame-0 COLMAP point cloud of a scene and return its path."""
    global COLMAP_KIND
    out_dir = os.path.join(raw_scene_dir(scene), "colmap0")
    dense_ply = os.path.join(out_dir, "points3d_dense.ply")
    sparse_ply = os.path.join(out_dir, "points3d_sparse.ply")
    final_ply = os.path.join(out_dir, "points3d.ply")
    kind_file = os.path.join(out_dir, "kind.txt")

    if os.path.exists(final_ply) and os.path.exists(kind_file):
        COLMAP_KIND = open(kind_file).read().strip()
        print("[%s] initial point cloud already built (%s)." % (scene, COLMAP_KIND))
        return final_ply

    install_colmap()
    os.makedirs(out_dir, exist_ok=True)
    work = os.path.join(out_dir, "work")
    shutil.rmtree(work, ignore_errors=True)
    sparse_in, images_dir = _write_colmap_text_model(scene, work)
    db = os.path.join(work, "database.db")

    _sh('colmap feature_extractor --database_path "%s" --image_path "%s" '
        '--ImageReader.camera_model PINHOLE --ImageReader.single_camera 1 '
        '--SiftExtraction.max_image_size 2048 --SiftExtraction.use_gpu 0' % (db, images_dir))
    _sh('colmap exhaustive_matcher --database_path "%s" --SiftMatching.use_gpu 0' % db)
    tri = os.path.join(work, "triangulated", "sparse")
    os.makedirs(tri, exist_ok=True)
    _sh('colmap point_triangulator --database_path "%s" --image_path "%s" '
        '--input_path "%s" --output_path "%s" --clear_points 1'
        % (db, images_dir, sparse_in, tri))
    _sh('colmap model_converter --input_path "%s" --output_path "%s" --output_type PLY'
        % (tri, sparse_ply))

    kind = "sparse"
    if COLMAP_DENSE:
        try:
            dense = os.path.join(work, "dense")
            os.makedirs(dense, exist_ok=True)
            _sh('colmap image_undistorter --image_path "%s" --input_path "%s" '
                '--output_path "%s"' % (images_dir, tri, dense))
            _sh('colmap patch_match_stereo --workspace_path "%s"' % dense)
            _sh('colmap stereo_fusion --workspace_path "%s" --output_path "%s"'
                % (dense, dense_ply))
            kind = "dense"
        except AssertionError as exc:
            print("[%s] dense fusion unavailable (%r); falling back to the sparse "
                  "triangulated points." % (scene, exc))

    _xyz, _rgb = read_ply(dense_ply if kind == "dense" else sparse_ply)
    write_ply(final_ply, _xyz, _rgb)
    with open(os.path.join(out_dir, "num_points.txt"), "w") as handle:
        handle.write(str(len(_xyz)))
    print("[%s] %d points in the initial cloud." % (scene, len(_xyz)))
    with open(kind_file, "w") as handle:
        handle.write(kind)
    shutil.rmtree(work, ignore_errors=True)
    COLMAP_KIND = kind
    print("[%s] initial point cloud: %s (%s)" % (scene, final_ply, kind))
    return final_ply


def point_cloud_size(scene):
    """Number of points in the shared initial cloud, or None if it is not built yet.

    This number matters: each method treats its own `num_pts` as a *cap*, not a
    target, so a small cloud is a small initialisation and nothing upsamples it.
    It is written into every benchmark JSON as `init_num_points`, next to
    `init_point_cloud` (dense or sparse), so two runs are never compared across
    initialisations of very different size.
    """
    path = os.path.join(raw_scene_dir(scene), "colmap0", "num_points.txt")
    if not os.path.exists(path):
        return None
    return int(open(path).read().strip())


print("COLMAP helpers ready (dense fusion requested: %s)." % COLMAP_DENSE)
print("Note: each method caps the cloud at its own num_pts but never upsamples it,")
print("so the size of this one cloud is the initialisation of all three.")

## 3.5 Per-scene preparation: N3DV in the Panoptic format

This is the method that needs most adaptation, and every addition is listed here so that none of it is silent.

The authors run on **Panoptic Sports**, a dataset they prepared themselves and released ready to use; the preparation code and the novel-view evaluation code are, in their own words, *not yet released*. Their loader expects `train_meta.json`, `ims/<cam>/<t>.png`, `seg/<cam>/<t>.png` and `init_pt_cld.npz`. The cell below builds exactly those files from N3DV.

**1. Cameras.** `poses_bounds.npy` gives the LLFF poses; they are converted to the OpenCV convention the repository documents (the same convention COLMAP uses), and the held-out `cam00` goes into `test_meta.json` while the other twenty go into `train_meta.json`. The dataset's own held-out camera is kept, so this notebook's test split is the same 50 views as the other three.

**2. World frame.** `get_loss` contains a **floor loss**, `clamp(y, min=0)`, which the README explains by "we know the ground-plane of the scenes we are using". Panoptic is y-down with the floor at y = 0, so the loss penalises geometry *below* the floor and is inactive for a well-placed scene. The world is therefore rotated so that +y points down (from the mean camera up-vector) and translated so that the lowest point of the initial cloud sits at y = 0. `get_dataset` also hard-codes `near = 1.0`, so the world is scaled to put the LLFF near bound at 1. All three are a change of units and orientation only: no relative geometry is altered.

**3. Segmentation masks.** The method needs a binary dynamic/static mask per image: its `seg` loss supervises them, and `is_fg` splits the primitives into the ones the rigidity losses act on and the background ones that are anchored in place. N3DV has no masks. They are estimated per camera as the pixels that differ from that camera's **temporal median** by more than `SEG_DIFF_THRESHOLD`, cleaned with a morphological open/close and a minimum-component-area filter. This is the largest addition of the four notebooks and is declared as a limitation: the mask quality bounds what the method can do here.

**4. Initial point cloud.** `init_pt_cld.npz` holds `[x, y, z, r, g, b, seg]`. The points and colours come from the **shared COLMAP stage** of cell 3.4, the same cloud notebooks `05` and `06` start from. A point is labelled foreground when it projects inside the per-camera union-over-time dynamic mask in more than half of the cameras that see it.

**5. What is *not* changed.** The losses, their weights, the densification, the learning rates, the optimiser, the per-frame initialisation by constant-velocity extrapolation and the two-phase schedule are upstream. The only training-loop constant touched is the per-timestep iteration count (cell 2), and only when the budget knobs are set.

In [ ]:
# 3.5  Per-scene preparation (Dynamic 3D Gaussians, N3DV -> Panoptic format).
import glob
import json
import os
import shutil

import cv2
import numpy as np
from PIL import Image

os.chdir(REPO_DIR)


def d3dg_world_frame(scene):
    """World transform that satisfies the two assumptions baked into the code.

    Returns (M, offset, scale) with p' = scale * (M @ p) + offset, chosen so that
    +y points down (the floor loss assumes a y-down frame with the ground at 0)
    and the LLFF near bound maps to 1 (get_dataset hard-codes near = 1.0).
    The camera-to-world matrices are mapped with the same transform.
    """
    c2w, focal, width, height, bounds = llff_poses(scene)
    # Mean camera up-vector in world coordinates. In the OpenCV convention the second
    # camera axis points down, so its mean over the rig is the world "down" direction.
    down = c2w[:, :3, 1].mean(axis=0)
    down = down / np.linalg.norm(down)
    # Rotation taking `down` onto +y
    target = np.array([0.0, 1.0, 0.0])
    v = np.cross(down, target)
    c = float(np.dot(down, target))
    if np.linalg.norm(v) < 1e-8:
        rot = np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    else:
        kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        rot = np.eye(3) + kmat + kmat @ kmat * ((1 - c) / (np.linalg.norm(v) ** 2))
    scale = 1.0 / (0.95 * float(bounds.min()))
    return rot, scale, focal, width, height, bounds


def apply_world(points, rot, scale, offset):
    return scale * (points @ rot.T) + offset


def median_and_masks(scene, cam):
    """Per-frame dynamic masks of one camera, from its temporal median.

    N3DV provides no segmentation. Everything that differs from the static
    background of that camera by more than SEG_DIFF_THRESHOLD is called dynamic;
    the result is cleaned morphologically and small components are dropped.
    Returns (masks, union) as uint8 arrays with values 0 and 1.
    """
    raw = raw_scene_dir(scene)
    frames = []
    for i in range(NUM_FRAMES):
        img = cv2.imread(os.path.join(raw, cam, "frames", "%04d.png" % i))
        frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0)
    stack = np.stack(frames, axis=0)
    median = np.median(stack, axis=0)

    height, width = median.shape[:2]
    min_area = int(SEG_MIN_AREA_RATIO * height * width)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    masks = []
    for frame in stack:
        diff = np.abs(frame - median).mean(axis=2)
        mask = (diff > SEG_DIFF_THRESHOLD).astype(np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        cleaned = np.zeros_like(mask)
        for label in range(1, n):
            if stats[label, cv2.CC_STAT_AREA] >= min_area:
                cleaned[labels == label] = 1
        masks.append(cleaned)
    masks = np.stack(masks, axis=0)
    union = (masks.max(axis=0) > 0).astype(np.uint8)
    return masks, union


def convert_scene(scene):
    """Write train_meta.json, test_meta.json, ims/, seg/ and init_pt_cld.npz."""
    src = source_path_for(scene)
    if os.path.exists(os.path.join(src, "init_pt_cld.npz")) and \
            os.path.exists(os.path.join(src, "test_meta.json")):
        print("[%s] already converted." % scene)
        return src

    raw = raw_scene_dir(scene)
    cams = n3dv_cameras(scene)
    c2w, focal, width, height, bounds = llff_poses(scene)
    rot, scale, focal, width, height, bounds = d3dg_world_frame(scene)

    # Apply the rotation and scale to the cameras, then choose the offset from the cloud.
    centres = apply_world(c2w[:, :3, 3], rot, scale, np.zeros(3))
    rotations = np.einsum("ij,njk->nik", rot, c2w[:, :3, :3])

    cloud = ensure_point_cloud(scene)
    xyz, rgb = read_ply(cloud)
    xyz_w = apply_world(xyz, rot, scale, np.zeros(3))
    # The "floor" is the lowest physical point, i.e. the largest y in a y-down frame.
    floor_y = float(np.percentile(xyz_w[:, 1], 99.0))
    offset = np.array([0.0, -floor_y, 0.0])
    xyz_w = xyz_w + offset
    centres = centres + offset

    intrinsics = np.array([[focal, 0.0, width / 2.0],
                           [0.0, focal, height / 2.0],
                           [0.0, 0.0, 1.0]])

    # Masks and image links
    ims_root = os.path.join(src, "ims")
    seg_root = os.path.join(src, "seg")
    unions = {}
    for cam in cams:
        os.makedirs(os.path.join(ims_root, cam), exist_ok=True)
        os.makedirs(os.path.join(seg_root, cam), exist_ok=True)
        masks, union = median_and_masks(scene, cam)
        unions[cam] = union
        for i in range(NUM_FRAMES):
            link = os.path.join(ims_root, cam, "%d.png" % i)
            if not os.path.exists(link):
                os.symlink(os.path.abspath(os.path.join(raw, cam, "frames", "%04d.png" % i)),
                           link)
            # The loader reads the mask as a float array and uses it directly as a
            # probability, so the PNG holds 0 and 1, not 0 and 255.
            Image.fromarray(masks[i], mode="L").save(
                os.path.join(seg_root, cam, "%d.png" % i))
        print("[%s/%s] dynamic pixels: %.2f%% of the frame on average"
              % (scene, cam, 100.0 * masks.mean()), flush=True)

    # Metadata, one entry per timestep and camera
    def meta_for(selected):
        indices = [cams.index(c) for c in selected]
        w2c_all, k_all, fn_all, id_all = [], [], [], []
        for t in range(NUM_FRAMES):
            w2c_t, k_t, fn_t, id_t = [], [], [], []
            for slot, index in enumerate(indices):
                pose = np.eye(4)
                pose[:3, :3] = rotations[index]
                pose[:3, 3] = centres[index]
                w2c_t.append(np.linalg.inv(pose).tolist())
                k_t.append(intrinsics.tolist())
                fn_t.append("%s/%d.png" % (cams[index], t))
                id_t.append(slot)
            w2c_all.append(w2c_t)
            k_all.append(k_t)
            fn_all.append(fn_t)
            id_all.append(id_t)
        return {"w": int(width), "h": int(height), "k": k_all, "w2c": w2c_all,
                "fn": fn_all, "cam_id": id_all}

    train_cams = [c for c in cams if c != HELD_OUT_CAMERA]
    test_cams = [c for c in cams if c == HELD_OUT_CAMERA]
    assert test_cams, "the held-out camera %s is not in the scene" % HELD_OUT_CAMERA
    assert len(train_cams) <= 50, "the code allocates per-camera colour corrections for 50 cameras"

    with open(os.path.join(src, "train_meta.json"), "w") as handle:
        json.dump(meta_for(train_cams), handle)
    with open(os.path.join(src, "test_meta.json"), "w") as handle:
        json.dump(meta_for(test_cams), handle)

    # Foreground label of every initial point: does it project inside the dynamic region
    # of more than half of the training cameras that see it?
    votes = np.zeros(len(xyz_w))
    seen = np.zeros(len(xyz_w))
    homogeneous = np.concatenate([xyz_w, np.ones((len(xyz_w), 1))], axis=1)
    for cam in train_cams:
        index = cams.index(cam)
        pose = np.eye(4)
        pose[:3, :3] = rotations[index]
        pose[:3, 3] = centres[index]
        cam_pts = (np.linalg.inv(pose) @ homogeneous.T).T[:, :3]
        valid = cam_pts[:, 2] > 1e-6
        uv = (intrinsics @ cam_pts.T).T
        with np.errstate(invalid="ignore", divide="ignore"):
            u = np.where(valid, uv[:, 0] / uv[:, 2], -1).astype(np.int64)
            v = np.where(valid, uv[:, 1] / uv[:, 2], -1).astype(np.int64)
        inside = valid & (u >= 0) & (u < int(width)) & (v >= 0) & (v < int(height))
        seen += inside
        idx = np.where(inside)[0]
        votes[idx] += unions[cam][v[idx], u[idx]]
    seg = (votes > 0.5 * np.maximum(seen, 1)).astype(np.float64)
    seg[seen == 0] = 0.0

    data = np.concatenate([xyz_w, rgb / 255.0, seg[:, None]], axis=1)
    np.savez(os.path.join(src, "init_pt_cld.npz"), data=data)
    print("[%s] %d initial points, %.1f%% foreground; %d train cameras, %d test"
          % (scene, len(data), 100.0 * seg.mean(), len(train_cams), len(test_cams)))
    return src


def prepare_scene(scene, mode):
    """Convert the scene; this method is driven by the launcher, not by flags."""
    ensure_scene_raw(scene)
    convert_scene(scene)
    return EXTRA_TRAIN_ARGS


def build_train_command(p, budget, extra):
    cmd = 'python train_one.py "%s" "%s"' % (p["scene"], p["run_name"])
    return cmd + ((' ' + extra) if extra else '')


def extra_train_env(p, budget):
    """Budget knobs and the destination the launcher moves the model to."""
    return {"BENCH_D3DG_ITERS_FIRST": str(int(D3DG_ITERS_FIRST)),
            "BENCH_D3DG_ITERS_PER_TS": str(int(D3DG_ITERS_PER_TIMESTEP)),
            "BENCH_D3DG_MODEL_PATH": p["model_path"]}


def extra_bench_config(p):
    return {"init_point_cloud": COLMAP_KIND,
            "init_num_points": point_cloud_size(p["scene"]),
            "budget_mode": D3DG_BUDGET_MODE,
            "iters_first_timestep": int(D3DG_ITERS_FIRST),
            "iters_per_timestep": int(D3DG_ITERS_PER_TIMESTEP),
            "seg_diff_threshold": float(SEG_DIFF_THRESHOLD),
            "seg_min_area_ratio": float(SEG_MIN_AREA_RATIO),
            "sequence": p["scene"]}


def release_scene(scene):
    shutil.rmtree(source_path_for(scene), ignore_errors=True)
    drop_scene_frames(scene)


def _folder_size_mb(root):
    total = 0
    for dirpath, _d, filenames in os.walk(root):
        for fn in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, fn))
            except OSError:
                pass
    return total / 1048576.0


def _sum_patterns(model_path, patterns):
    files = []
    for pat in patterns:
        files += sorted(glob.glob(os.path.join(model_path, pat)))
    files = [f for f in files if os.path.isfile(f)]
    total = sum(os.path.getsize(f) for f in files)
    return files, total


# The model is a single params.npz: the per-Gaussian attributes that do not change after
# the first frame are stored once, and means3D, rgb_colors and unnorm_rotations are stored
# per frame. Unlike the other three methods the file therefore grows with the window,
# which is a property of the representation and is discussed in the results.
REQUIRED_PATTERNS = ("params.npz",)


def model_storage_report(model_path):
    files, total = _sum_patterns(model_path, REQUIRED_PATTERNS)
    report = {
        "required_files": [os.path.relpath(f, model_path) for f in files],
        "model_storage_mb": round(total / 1048576.0, 4),
        "full_folder_mb": round(_folder_size_mb(model_path), 4),
    }
    path = os.path.join(model_path, "params.npz")
    if os.path.exists(path):
        import numpy as _np

        with _np.load(path) as blob:
            report["arrays"] = {k: list(blob[k].shape) for k in blob.files}
    return report


print("Preparation ready.")
print("Window     : %d frames, held-out camera %s" % (NUM_FRAMES, HELD_OUT_CAMERA))
print("Budget     : %s, %d + %d x %d = %d steps"
      % (D3DG_BUDGET_MODE, D3DG_ITERS_FIRST, D3DG_ITERS_PER_TIMESTEP,
         NUM_FRAMES - 1, MAX_ITERATIONS))
print("Masks      : threshold %.3f, minimum component %.4f of the frame"
      % (SEG_DIFF_THRESHOLD, SEG_MIN_AREA_RATIO))

## 4. Training with incremental benchmarking

**Instrumentation.** The training loop of each repository calls one of its own functions at every iteration. Instead of rewriting the loop, which would alter densification schedules, learning-rate ramps and stage handling, the notebook appends a few lines to `train.py` that wrap that function with a monitor. The upstream code path, hyper-parameters and command line stay untouched, a pristine copy is kept as `train.py.orig`, and without the `BENCH_CONFIG` environment variable `train.py` behaves exactly as upstream.

Two of the four repositories expose a `training_report(...)` that already receives the loop state, as in the monocular study. The other two do not have such a function at all: their loops call a narrower per-iteration function, so the monitor recovers the loop state from the **calling frame** instead of from the signature. The frame is only read, never written, and the wrapped function is still an upstream function the loop already called at every iteration.

**The monitor core is byte-identical in notebooks `04`–`07`** — it is delimited by the `BENCHMARK MONITOR CORE` markers — and everything that differs between repositories lives in the glue appended after it. At every sampling point the monitor evaluates the current model on the **full 50-view test split** (`cam00`, frames 0–49) with each repository's own metric modules and records PSNR, SSIM, LPIPS (VGG), the evaluation L1, the training time net of the benchmark overhead, the iteration count, the images seen, the number of Gaussians and the evaluation resolution. The JSON summary is rewritten atomically after each evaluation, so an interrupted run still leaves a valid file.

**Protocol B cost.** A run stops only after the target has held for `TARGET_CONSECUTIVE_HITS` consecutive samples, so the termination time overstates the cost, by a different amount for each method. The figure to report is the **first crossing**, extracted by cell 4.4.

In [ ]:
# 4.0  Install the benchmark monitor and build the training driver
import json
import os
import shutil
import subprocess
import sys
import threading
import time

os.chdir(REPO_DIR)

MONITOR_SRC = r'''# benchmark_monitor.py -- generated by notebook 04: incremental benchmark logger for
# Dynamic 3D Gaussians (Luiten et al.) on Neural 3D Video.
# This repository has no training_report(...): its loop calls report_progress(...) at every
# iteration, so that is the function the notebook wraps, and the loop state that the narrower
# signature does not carry (the current frame index, the per-frame snapshots taken so far, the
# sequence name and the current loss) is read from the calling frame.

# --- repository metric modules: the only import block that differs between notebooks 04-07 ---
from external import calc_psnr as _repo_psnr
from external import calc_ssim as _repo_ssim
from helpers import l1_loss_v1 as _repo_l1

# === BENCHMARK MONITOR CORE (byte-identical in notebooks 04-07) ===
# Everything between this marker and END BENCHMARK MONITOR CORE is shared verbatim by the
# four multi-view notebooks. The repository-specific glue is appended after it and provides
# _stage_is_evaluable, _global_iteration, _num_gaussians, _train_batch_loss, _force_eval,
# _iter_eval_views, _render_pair, _save_model and _maybe_track_best.

import atexit
import inspect
import json
import os
import sys
import time

import torch


# Configuration written by the notebook, passed through $BENCH_CONFIG

_CFG_PATH = os.environ.get("BENCH_CONFIG", "")

with open(_CFG_PATH, "r") as _f:
    CFG = json.load(_f)

METHOD = CFG["method"]
SCENE = CFG["scene"]
MODE = CFG["mode"]                                   # "iterations" | "target_eval_loss"
JSON_PATH = CFG["json_path"]

EVAL_EVERY_N_ITERS = int(CFG.get("eval_every_n_iters", 2000))
EVAL_EVERY_N_SECONDS = float(CFG.get("eval_every_n_seconds", 600.0))
MAX_ITERATIONS = int(CFG.get("max_iterations", 30000))
SAFETY_MAX_ITERATIONS = int(CFG.get("safety_max_iterations", 100000))

# Protocol B stopping criterion: "eval_loss" (eval_loss <= target) or "psnr" (psnr >= target)
TARGET_METRIC = CFG.get("target_metric", "psnr")
TARGET_PSNR = float(CFG.get("target_psnr", 30.0))
TARGET_EVAL_LOSS = float(CFG.get("target_eval_loss", 0.0))
MIN_ITERATIONS_BEFORE_STOP = int(CFG.get("min_iterations_before_stop", 2000))
# Test metrics are not monotonic: the target must hold for several consecutive samples.
TARGET_CONSECUTIVE_HITS = int(CFG.get("target_consecutive_hits", 2))

# Evaluation loss: a common photometric yardstick, not any method's own objective
# (regularisers never look at the ground truth and have no evaluation counterpart).
EVAL_LOSS_KIND = CFG.get("eval_loss_kind", "l1")     # "l1" | "l1_dssim"
LAMBDA_DSSIM = float(CFG.get("lambda_dssim", 0.2))
LPIPS_NET = CFG.get("lpips_net", "vgg")
# "lpips-pip" | "lpipsPyTorch" | "auto". The multi-view study forces one backend for the four
# methods, because Dynamic 3D Gaussians does not bundle lpipsPyTorch and a mixed backend would
# put a systematic offset on the LPIPS column.
LPIPS_BACKEND = CFG.get("lpips_backend", "auto")
MAX_EVAL_VIEWS = int(CFG.get("max_eval_views", 0))   # 0 = all test views
EVAL_AT_FIRST_ITERATION = bool(CFG.get("eval_at_first_iteration", True))

# BATCH_SIZE: views consumed per optimiser step (per-method, from the official config).
# ITERATION_OFFSET: steps of an earlier stage whose counter restarted (coarse stage of Wu et al.).
BATCH_SIZE = int(CFG.get("batch_size", 1))
ITERATION_OFFSET = int(CFG.get("iteration_offset", 0))

# Multi-view protocol: the temporal window and the held-out camera, recorded in every entry so
# that a JSON file is self-describing and two runs can never be compared across windows.
NUM_FRAMES = int(CFG.get("num_frames", 0))
HELD_OUT_CAMERA = CFG.get("held_out_camera", "cam00")
DATASET = CFG.get("dataset", "n3dv")


# LPIPS: one backend for the whole study; the model is built once.

_lpips_model = None
_lpips_backend = None


def _build_lpips_pip():
    import lpips as _lpips_pkg
    return _lpips_pkg.LPIPS(net=LPIPS_NET).to("cuda").eval(), "lpips-pip"


def _build_lpips_bundled():
    from lpipsPyTorch.modules.lpips import LPIPS
    return LPIPS(net_type=LPIPS_NET).to("cuda").eval(), "lpipsPyTorch"


def _get_lpips():
    global _lpips_model, _lpips_backend
    if _lpips_model is not None:
        return _lpips_model, _lpips_backend
    if LPIPS_BACKEND == "lpips-pip":
        _lpips_model, _lpips_backend = _build_lpips_pip()
    elif LPIPS_BACKEND == "lpipsPyTorch":
        _lpips_model, _lpips_backend = _build_lpips_bundled()
    else:
        try:
            _lpips_model, _lpips_backend = _build_lpips_bundled()
        except Exception as exc:
            print("[benchmark] lpipsPyTorch unavailable (%r), falling back to the "
                  "lpips pip package." % (exc,))
            _lpips_model, _lpips_backend = _build_lpips_pip()
    _SUMMARY["lpips_backend"] = _lpips_backend
    print("[benchmark] LPIPS backend: %s (%s)" % (_lpips_backend, LPIPS_NET), flush=True)
    return _lpips_model, _lpips_backend


def _lpips_value(image, gt):
    model, backend = _get_lpips()
    x = image.unsqueeze(0)
    y = gt.unsqueeze(0)
    if backend == "lpips-pip":
        # the pip package expects inputs in [-1, 1]
        x = x * 2.0 - 1.0
        y = y * 2.0 - 1.0
    return model(x, y).mean().item()


# JSON summary, rewritten atomically at every evaluation

_SUMMARY = {
    "method": METHOD,
    "scene": SCENE,
    "mode": MODE,
    "dataset": DATASET,
    "num_frames": NUM_FRAMES,
    "held_out_camera": HELD_OUT_CAMERA,
    "config": CFG,
    "status": "running",
    "target_reached": False,
    "entries": [],
    "final": None,
}


def _flush_summary():
    directory = os.path.dirname(JSON_PATH)
    if directory:
        os.makedirs(directory, exist_ok=True)
    tmp_path = JSON_PATH + ".tmp"
    with open(tmp_path, "w") as handle:
        json.dump(_SUMMARY, handle, indent=2)
    os.replace(tmp_path, JSON_PATH)


@atexit.register
def _on_exit():
    if _SUMMARY["status"] == "running":
        _SUMMARY["status"] = "finished"
    if _SUMMARY["entries"]:
        _SUMMARY["final"] = _SUMMARY["entries"][-1]
    try:
        _flush_summary()
    except Exception as exc:
        print("[benchmark] could not write the JSON summary: %r" % (exc,))


class _Monitor(object):

    def __init__(self):
        self.t_start = None
        self.eval_overhead_s = 0.0
        self.last_eval_time = None
        self.last_eval_iteration = -1
        self.best_psnr = -1.0
        self.n_evals = 0
        self.consecutive_hits = 0

    def _target_met(self, entry):
        if entry["iteration"] < MIN_ITERATIONS_BEFORE_STOP:
            return False
        if TARGET_METRIC == "psnr":
            return entry["psnr"] >= TARGET_PSNR
        return entry["eval_loss"] <= TARGET_EVAL_LOSS

    def _must_evaluate(self, a, iteration, now):
        if iteration == self.last_eval_iteration:
            return False
        if EVAL_AT_FIRST_ITERATION and self.n_evals == 0:
            return True
        if _force_eval(a):
            return True
        if MODE == "iterations":
            if iteration >= MAX_ITERATIONS:
                return True
            return (iteration % EVAL_EVERY_N_ITERS) == 0
        # MODE == "target_eval_loss": time-driven sampling
        if iteration >= SAFETY_MAX_ITERATIONS:
            return True
        return (now - self.last_eval_time) >= EVAL_EVERY_N_SECONDS

    @torch.no_grad()
    def _evaluate(self, a):
        psnr_sum, ssim_sum, lpips_sum, l1_sum, n_views = 0.0, 0.0, 0.0, 0.0, 0
        eval_h, eval_w = 0, 0
        for view in _iter_eval_views(a):
            image, gt = _render_pair(a, view)
            eval_h, eval_w = int(image.shape[-2]), int(image.shape[-1])
            image_b = image.unsqueeze(0)
            gt_b = gt.unsqueeze(0)
            psnr_sum += _repo_psnr(image_b, gt_b).mean().item()
            ssim_sum += _repo_ssim(image, gt).mean().item()
            l1_sum += _repo_l1(image, gt).mean().item()
            lpips_sum += _lpips_value(image, gt)
            n_views += 1
        n_views = max(n_views, 1)
        return {
            "psnr": psnr_sum / n_views,
            "ssim": ssim_sum / n_views,
            "lpips": lpips_sum / n_views,
            "eval_l1_loss": l1_sum / n_views,
            "num_eval_views": n_views,
            "eval_resolution": [eval_h, eval_w],
        }

    def step(self, a):
        now = time.time()
        if self.t_start is None:
            self.t_start = now
            self.last_eval_time = now

        iteration = int(_global_iteration(a))
        stage = _stage_of(a)

        if not _stage_is_evaluable(stage):
            return
        if not self._must_evaluate(a, iteration, now):
            return

        eval_t0 = time.time()
        torch.cuda.empty_cache()
        metrics = self._evaluate(a)
        torch.cuda.synchronize()
        eval_dt = time.time() - eval_t0

        # Both loss variants are always recorded (SSIM is computed anyway)
        eval_photometric = ((1.0 - LAMBDA_DSSIM) * metrics["eval_l1_loss"]
                            + LAMBDA_DSSIM * (1.0 - metrics["ssim"]))
        eval_loss = (eval_photometric if EVAL_LOSS_KIND == "l1_dssim"
                     else metrics["eval_l1_loss"])

        wall = eval_t0 - self.t_start
        entry = {
            "iteration": iteration,
            "total_iterations": iteration + ITERATION_OFFSET,
            "images_seen": (iteration + ITERATION_OFFSET) * BATCH_SIZE,
            "stage": stage if stage is not None else "single_stage",
            "training_time_s": wall - self.eval_overhead_s,
            "wall_time_s": wall,
            "benchmark_overhead_s": self.eval_overhead_s,
            "psnr": metrics["psnr"],
            "ssim": metrics["ssim"],
            "lpips": metrics["lpips"],
            "eval_l1_loss": metrics["eval_l1_loss"],
            "eval_photometric_loss": eval_photometric,
            "eval_loss": eval_loss,
            "eval_loss_kind": EVAL_LOSS_KIND,
            "num_eval_views": metrics["num_eval_views"],
            "eval_resolution": metrics["eval_resolution"],
            "num_gaussians": int(_num_gaussians(a)),
            "train_batch_loss": _train_batch_loss(a),
            "eval_duration_s": eval_dt,
        }
        entry.update(_extra_entry_fields(a))

        self.last_eval_iteration = iteration
        self.n_evals += 1

        _SUMMARY["entries"].append(entry)
        _SUMMARY["final"] = entry
        _flush_summary()

        print("\n[benchmark] iter %7d | train %8.1fs | PSNR %6.3f | SSIM %.4f | "
              "LPIPS %.4f | eval_loss %.5f | #G %d | eval@%dx%d on %d views"
              % (iteration, entry["training_time_s"], entry["psnr"], entry["ssim"],
                 entry["lpips"], entry["eval_loss"], entry["num_gaussians"],
                 entry["eval_resolution"][1], entry["eval_resolution"][0],
                 entry["num_eval_views"]),
              flush=True)

        _maybe_track_best(a, entry, self)

        # Evaluation, JSON flush and best-checkpoint writes are benchmark overhead, not training time.
        self.eval_overhead_s += time.time() - eval_t0
        self.last_eval_time = time.time()

        if MODE == "target_eval_loss":
            if self._target_met(entry):
                self.consecutive_hits += 1
                if TARGET_METRIC == "psnr":
                    print("[benchmark] target met (%d/%d consecutive): PSNR %.4f >= %.4f"
                          % (self.consecutive_hits, TARGET_CONSECUTIVE_HITS,
                             entry["psnr"], TARGET_PSNR), flush=True)
                else:
                    print("[benchmark] target met (%d/%d consecutive): eval_loss "
                          "%.6f <= %.6f" % (self.consecutive_hits,
                                            TARGET_CONSECUTIVE_HITS,
                                            eval_loss, TARGET_EVAL_LOSS), flush=True)
            else:
                if self.consecutive_hits:
                    print("[benchmark] target lost again, consecutive counter reset.",
                          flush=True)
                self.consecutive_hits = 0

            if self.consecutive_hits >= TARGET_CONSECUTIVE_HITS:
                criterion = ("PSNR >= %.4f" % TARGET_PSNR if TARGET_METRIC == "psnr"
                             else "eval_loss <= %.6f" % TARGET_EVAL_LOSS)
                print("\n[benchmark] TARGET REACHED (%s held for %d consecutive "
                      "sampling points) at iteration %d. Saving the model and "
                      "stopping training." % (criterion, TARGET_CONSECUTIVE_HITS,
                                              iteration), flush=True)
                _SUMMARY["target_reached"] = True
                _SUMMARY["status"] = "stopped_on_target"
                _SUMMARY["target_metric"] = TARGET_METRIC
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)
            if iteration >= SAFETY_MAX_ITERATIONS:
                print("\n[benchmark] Safety cap of %d iterations reached without "
                      "hitting the target eval loss. Saving and stopping."
                      % SAFETY_MAX_ITERATIONS, flush=True)
                _SUMMARY["status"] = "stopped_on_safety_cap"
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)


_MONITOR = _Monitor()


# Hook factory, called from the block injected at the bottom of train.py

def make_hook(original_hook):
    signature = inspect.signature(original_hook)

    def hooked(*args, **kwargs):
        result = original_hook(*args, **kwargs)
        try:
            bound = signature.bind(*args, **kwargs)
            bound.apply_defaults()
            arguments = dict(bound.arguments)
            arguments["_caller_locals"] = _caller_locals()
            _MONITOR.step(arguments)
        except SystemExit:
            raise
        except Exception as exc:
            print("[benchmark] evaluation hook failed: %r" % (exc,), flush=True)
        return result

    print("[benchmark] monitor installed (method=%s, scene=%s, mode=%s, frames=%d)"
          % (METHOD, SCENE, MODE, NUM_FRAMES), flush=True)
    return hooked


def _caller_locals(depth=2):
    """Locals of the training loop that called the wrapped function.

    Two of the four repositories pass everything the monitor needs as arguments
    (as `training_report(...)` does); the other two call a per-iteration function
    with a narrower signature, and the loop state has to be read from the calling
    frame instead. The frame is only read, never written.
    """
    # depth 0 = this function, 1 = the wrapper in make_hook, 2 = the training loop.
    frame = sys._getframe(depth)
    return frame.f_locals


def _extra_entry_fields(a):
    """Per-method extra columns; the glue overrides this when it has any."""
    return {}


# === END BENCHMARK MONITOR CORE ===

# Repository-specific glue: JonathonLuiten/Dynamic3DGaussians.
#
# Two things make this method different from the other three, and both are handled here
# rather than in the shared core.
#
# 1. The iteration counter restarts at every frame. report_progress(params, data, i, ...)
#    receives the *per-frame* index, so the global step is recomputed from the frame index
#    and the two per-frame budgets.
# 2. The model is a sequence of per-frame snapshots, not one state. To make the test metric
#    mean exactly what it means for the other three -- the 50 test views of the window,
#    rendered by the model as it stands right now -- a frame already finished is rendered
#    from its snapshot in output_params, and the current and future frames from the live
#    parameters, which is all the model can produce for them.

import json as _json

import numpy as _np
from PIL import Image as _Image

from helpers import setup_camera as _setup_camera
from helpers import params2rendervar as _params2rendervar
from helpers import params2cpu as _params2cpu
from helpers import save_params as _save_params
from diff_gaussian_rasterization import GaussianRasterizer as _Renderer

_TEST_CAMERAS = None
_TEST_IMAGES = None


def _load_test_split(sequence):
    """Cameras and ground truth of the held-out camera, one entry per frame.

    The images are cached on the CPU as uint8 (about 200 MB for a 50-frame window)
    and moved to the GPU one at a time, so the cache does not compete with training
    for VRAM.
    """
    global _TEST_CAMERAS, _TEST_IMAGES
    if _TEST_CAMERAS is not None:
        return _TEST_CAMERAS, _TEST_IMAGES
    with open("./data/%s/test_meta.json" % sequence) as handle:
        meta = _json.load(handle)
    cameras, images = [], []
    for t in range(len(meta["fn"])):
        # The test split has exactly one camera per frame: the held-out one.
        camera = _setup_camera(meta["w"], meta["h"], meta["k"][t][0],
                               meta["w2c"][t][0], near=1.0, far=100)
        array = _np.array(_Image.open("./data/%s/ims/%s" % (sequence, meta["fn"][t][0])))
        images.append(torch.tensor(array).permute(2, 0, 1).contiguous())
        cameras.append(camera)
    _TEST_CAMERAS, _TEST_IMAGES = cameras, images
    print("[benchmark] test split: %d views of the held-out camera" % len(cameras),
          flush=True)
    return _TEST_CAMERAS, _TEST_IMAGES


def _sequence_of(a):
    return a["_caller_locals"]["seq"]


def _timestep_of(a):
    return int(a["_caller_locals"]["t"])


def _stage_is_evaluable(stage):
    return True


def _stage_of(a):
    return "t%03d" % _timestep_of(a)


def _global_iteration(a):
    """Cumulative optimiser steps since the beginning of the sequence.

    report_progress receives i, the index inside the current frame, which restarts
    at every frame; the shared core needs a monotone counter.
    """
    t = _timestep_of(a)
    i = int(a["i"])
    first = int(CFG.get("iters_first_timestep", 10000))
    per = int(CFG.get("iters_per_timestep", 2000))
    if t == 0:
        return i + 1
    return first + (t - 1) * per + i + 1


def _num_gaussians(a):
    return a["params"]["means3D"].shape[0]


def _train_batch_loss(a):
    loss = a.get("_caller_locals", {}).get("loss", None)
    return float(loss.item()) if hasattr(loss, "item") else None


def _force_eval(a):
    """Always sample at the end of a frame, whatever the iteration grid says."""
    t = _timestep_of(a)
    first = int(CFG.get("iters_first_timestep", 10000))
    per = int(CFG.get("iters_per_timestep", 2000))
    budget = first if t == 0 else per
    return int(a["i"]) == budget - 1


def _extra_entry_fields(a):
    return {"timesteps_done": _timestep_of(a),
            "iter_in_timestep": int(a["i"]) + 1,
            "num_timesteps": len(_TEST_CAMERAS or []) or None}


def _iter_eval_views(a):
    cameras, images = _load_test_split(_sequence_of(a))
    total = len(cameras)
    indices = list(range(total))
    if MAX_EVAL_VIEWS and MAX_EVAL_VIEWS < total:
        step = max(1, total // MAX_EVAL_VIEWS)
        indices = indices[::step][:MAX_EVAL_VIEWS]
    for index in indices:
        yield (index, cameras[index], images[index])


def _rendervar_at(a, t):
    """Render variables of frame t: its snapshot if it is finished, else the live model.

    params2cpu keeps every array for the first frame and only the three that change
    afterwards, so a later snapshot is completed with the current values of the
    attributes the method freezes after the first frame.
    """
    params = a["params"]
    finished = a["_caller_locals"].get("output_params", [])
    if t >= len(finished):
        return _params2rendervar(params)
    snapshot = finished[t]
    merged = {}
    for key, value in params.items():
        if key in snapshot:
            # Same device and dtype as the live parameter, so the two branches of this
            # function hand the rasterizer identical tensors. The keys that are not in a
            # later snapshot stay as the live nn.Parameter objects; params2rendervar
            # normalises, exponentiates or passes them through either way, and the
            # rasterizer already receives Parameters on the upstream path.
            merged[key] = torch.tensor(snapshot[key], device="cuda", dtype=value.dtype)
        else:
            merged[key] = value
    return _params2rendervar(merged)


def _render_pair(a, view):
    index, camera, gt_cpu = view
    rendervar = _rendervar_at(a, index)
    image, _radius, _depth = _Renderer(raster_settings=camera)(**rendervar)
    # The per-camera exposure correction cam_m / cam_c is learnt for the *training*
    # cameras only; the held-out camera has none, so the raw render is scored.
    image = torch.clamp(image, 0.0, 1.0)
    gt = torch.clamp(gt_cpu.to("cuda").float() / 255.0, 0.0, 1.0)
    return image, gt[0:3, :, :]


def _save_model(a):
    """Write params.npz from the frames finished so far plus the current one.

    Upstream only saves at the very end of the sequence; Protocol B stops in the
    middle, so the same file is written from the snapshots taken up to that point.
    """
    loop = a["_caller_locals"]
    finished = list(loop.get("output_params", []))
    finished.append(_params2cpu(a["params"], _timestep_of(a) == 0))
    if len(finished) < 2:
        print("[benchmark] only one frame finished; params.npz needs at least two, "
              "nothing written.", flush=True)
        return
    _save_params(finished, loop["seq"], loop["exp"])
    print("[benchmark] params.npz written with %d frames." % len(finished), flush=True)


def _maybe_track_best(a, entry, monitor):
    if entry["psnr"] > monitor.best_psnr:
        monitor.best_psnr = entry["psnr"]
        _SUMMARY["best_psnr"] = entry["psnr"]
        _SUMMARY["best_psnr_iteration"] = entry["iteration"]
'''

with open(os.path.join(REPO_DIR, "benchmark_monitor.py"), "w") as handle:
    handle.write(MONITOR_SRC)

# Inject the hook into train.py (idempotent; a pristine copy is kept as train.py.orig).
# report_progress() is defined in another module and imported into train.py's namespace, so
# rebinding the global name in train.py is enough: the loop resolves it at call time.
TRAIN_PY = os.path.join(REPO_DIR, "train.py")
BACKUP_PY = TRAIN_PY + ".orig"
if not os.path.exists(BACKUP_PY):
    shutil.copyfile(TRAIN_PY, BACKUP_PY)
shutil.copyfile(BACKUP_PY, TRAIN_PY)

with open(TRAIN_PY, "r") as handle:
    source = handle.read()

INJECTED = (
    "# === BENCHMARK HOOK (injected by the notebook) ===\n"
    "import os as _bench_os\n"
    "if _bench_os.environ.get('BENCH_CONFIG'):\n"
    "    import benchmark_monitor as _bench\n"
    "    report_progress = _bench.make_hook(report_progress)\n"
    "# === END BENCHMARK HOOK ===\n\n"
)
ANCHOR = 'if __name__ == "__main__":'
assert source.count(ANCHOR) == 1, "Unexpected train.py layout."
assert ("report_progress" in source), "report_progress is not referenced in train.py."
with open(TRAIN_PY, "w") as handle:
    handle.write(source.replace(ANCHOR, INJECTED + ANCHOR, 1))

print("benchmark_monitor.py written and the hook injected into train.py (report_progress).")
print("Without the BENCH_CONFIG environment variable train.py behaves exactly "
      "as upstream (pristine copy kept as train.py.orig).")


# Peak VRAM: training runs in a subprocess, so nvidia-smi is polled for the whole device.
class VramSampler(threading.Thread):

    def __init__(self, period_s=2.0):
        threading.Thread.__init__(self)
        self.daemon = True
        self.period_s = period_s
        self.peak_mb = 0.0
        self._stop_event = threading.Event()

    def run(self):
        while not self._stop_event.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"])
                self.peak_mb = max(self.peak_mb,
                                   float(out.decode().strip().splitlines()[0]))
            except Exception:
                pass
            self._stop_event.wait(self.period_s)

    def stop(self):
        self._stop_event.set()
        return self.peak_mb


def write_bench_config(p):
    """Serialise the benchmark settings read by benchmark_monitor.py."""
    target = TARGET_EVAL_LOSS_PER_SCENE.get(p["scene"])
    config = {
        "method": METHOD_NAME,
        "scene": p["scene"],
        "mode": p["mode"],
        "json_path": p["bench_json"],
        "eval_every_n_iters": int(EVAL_EVERY_N_ITERS),
        "eval_every_n_seconds": float(EVAL_EVERY_N_MINUTES) * 60.0,
        "max_iterations": int(MAX_ITERATIONS),
        "safety_max_iterations": int(SAFETY_MAX_ITERATIONS),
        "target_metric": TARGET_METRIC,
        "target_psnr": float(TARGET_PSNR),
        "target_eval_loss": float(target) if target is not None else 0.0,
        "min_iterations_before_stop": int(MIN_ITERATIONS_BEFORE_STOP),
        "target_consecutive_hits": int(TARGET_CONSECUTIVE_HITS),
        "eval_loss_kind": EVAL_LOSS_KIND,
        "lambda_dssim": float(LAMBDA_DSSIM),
        "lpips_net": LPIPS_NET,
        "lpips_backend": LPIPS_BACKEND,
        "max_eval_views": int(MAX_EVAL_VIEWS),
        "eval_at_first_iteration": bool(EVAL_AT_FIRST_ITERATION),
        "source_path": p["source_path"],
        "model_path": p["model_path"],
        "batch_size": int(BATCH_SIZE),
        "iteration_offset": int(globals().get("COARSE_ITERATIONS", 0)),
        "dataset": "n3dv",
        # A block of a run trained in blocks covers fewer frames than the study window.
        "num_frames": int(p.get("num_frames", NUM_FRAMES)),
        "held_out_camera": HELD_OUT_CAMERA,
        "image_downscale": int(IMAGE_DOWNSCALE),
    }
    config.update(extra_bench_config(p))
    os.makedirs(p["benchmark_dir"], exist_ok=True)
    with open(p["bench_config"], "w") as handle:
        json.dump(config, handle, indent=2)
    return config


def record_run_cost(p, wall_s, peak_vram_mb):
    if not os.path.exists(p["bench_json"]):
        return
    with open(p["bench_json"], "r") as handle:
        summary = json.load(handle)
    summary["process_wall_time_s"] = wall_s
    summary["peak_vram_mb_nvidia_smi"] = peak_vram_mb
    with open(p["bench_json"], "w") as handle:
        json.dump(summary, handle, indent=2)


def run_training_process(p, cmd, budget):
    """Run one training command with the monitor attached; return (returncode, wall_s, peak_mb).

    The loop needs this to be a function, so the training command is issued
    with subprocess rather than the `!` magic (which cannot appear inside a
    function body). Output is streamed line by line, as `!` would do.
    """
    print("=" * 78)
    print("[%s | %s] %s" % (p["scene"], p["mode"], cmd))
    print("=" * 78, flush=True)

    env = dict(os.environ, BENCH_CONFIG=p["bench_config"])
    env.update(extra_train_env(p, budget))
    sampler = VramSampler()
    sampler.start()
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    wall = time.time() - t0
    peak = sampler.stop()
    record_run_cost(p, wall, peak)
    return proc.returncode, wall, peak


def record_storage(p):
    """Measure the model storage of one run into its JSON, before the model is discarded."""
    try:
        storage = model_storage_report(p["model_path"])
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
        summary["storage"] = storage
        with open(p["bench_json"], "w") as handle:
            json.dump(summary, handle, indent=2)
        print("[%s] model storage %.2f MB (full folder %.2f MB)"
              % (p["scene"], storage["model_storage_mb"], storage["full_folder_mb"]))
    except Exception as exc:
        print("[%s] storage measurement failed: %r" % (p["scene"], exc))


def train_scene(scene, mode=None):
    """Train one scene under one protocol. Returns a short status string."""
    mode = mode or TRAINING_MODE
    # A method whose official unit is shorter than the study window (Spacetime Gaussians
    # beyond 50 frames) trains the scene as several independent blocks instead.
    if globals().get("NUM_BLOCKS", 1) > 1:
        return train_scene_in_blocks(scene, mode)
    p = activate_scene(scene, mode)

    if mode == "target_eval_loss" and TARGET_METRIC == "eval_loss":
        if TARGET_EVAL_LOSS_PER_SCENE.get(scene) is None:
            print("[%s] no target calibrated for this scene -> skipped." % scene)
            return "skipped_no_target"

    if SKIP_TRAINING_IF_TRAINED and run_is_complete(p):
        print("[%s] already benchmarked -> skipped (%s)."
              % (scene, p["bench_json"]))
        return "skipped_already_trained"

    os.chdir(REPO_DIR)
    extra = prepare_scene(scene, mode)
    write_bench_config(p)
    budget = MAX_ITERATIONS if mode == "iterations" else SAFETY_MAX_ITERATIONS
    cmd = build_train_command(p, budget, extra)
    returncode, wall, peak = run_training_process(p, cmd, budget)

    # Measure model storage now, before the local model is discarded.
    record_storage(p)

    if KEEP_MODEL_ON_DRIVE:
        print("[%s] model kept at %s" % (scene, p["model_path"]))
    else:
        shutil.rmtree(p["model_path"], ignore_errors=True)
        print("[%s] local model discarded; metrics kept on Drive." % scene)

    # The JSON is final at this point: put it on Drive before anything else can go wrong.
    try:
        sync_push(p["run_name"])
    except Exception as exc:
        print("[%s] sync failed: %r (results are still in %s)"
              % (scene, exc, p["benchmark_dir"]))

    if DELETE_FRAMES_AFTER_TRAIN:
        try:
            release_scene(scene)
        except Exception as exc:
            print("[%s] could not free the scene data: %r" % (scene, exc))

    print("\n[%s] finished in %dm %ds (return code %d), peak VRAM %.0f MB"
          % (scene, int(wall // 60), int(wall % 60), returncode, peak),
          flush=True)
    return "ok" if returncode == 0 else "failed(%d)" % returncode


print("Training driver ready.")

### 4.1 Single-scene training

Trains `SCENE` under the protocol selected by `TRAINING_MODE`. Inert unless `RUN_MODE == "single"`.

In [ ]:
# 4.1  Train a single scene (inert unless RUN_MODE == "single")
if RUN_MODE != "single":
    print("RUN_MODE is 'loop': use cell 4.2 instead.")
else:
    status = train_scene(SCENE)
    print("Status: %s" % status)

### 4.2 Training loop

Trains every scene in `SCENES_TO_RUN` sequentially, one run fully written to storage before the next starts. Inert unless `RUN_MODE == "loop"`. The loop is resumable: completed runs are skipped, so re-running the cell after a disconnection or a GPU-quota stop continues where it stopped. A scene that raises an exception is recorded and the loop moves on. Scenes run one at a time on purpose, since sharing the GPU would corrupt the time and VRAM measurements — and, in the multi-view setting, because the extracted frames of a single scene already fill most of the runtime disk.

In [ ]:
# 4.2  Train every scene in SCENES_TO_RUN (re-run the cell to resume after a disconnection)
import time

if RUN_MODE != "loop":
    print("RUN_MODE is 'single': use cell 4.1 instead.")
else:
    loop_results = {}
    loop_t0 = time.time()
    for _i, _scene in enumerate(SCENES_TO_RUN, 1):
        print("\n\n### [%d/%d] %s ###" % (_i, len(SCENES_TO_RUN), _scene),
              flush=True)
        try:
            loop_results[_scene] = train_scene(_scene)
        except KeyboardInterrupt:
            loop_results[_scene] = "interrupted"
            break
        except Exception as _exc:
            loop_results[_scene] = "error: %r" % (_exc,)
            print("[%s] ERROR: %r -- continuing with the next scene."
                  % (_scene, _exc), flush=True)

    print("\n" + "=" * 60)
    print(" LOOP SUMMARY (%s, %.1f min)"
          % (TRAINING_MODE, (time.time() - loop_t0) / 60.0))
    print("=" * 60)
    for _scene in SCENES_TO_RUN:
        print("  %-16s %s" % (_scene, loop_results.get(_scene, "not reached")))

### 4.3 Protocol B target calibration

Run after the Protocol A loop. Prints this method's final eval L1 per scene and the candidate target with a 5% margin. The target used in `TARGET_EVAL_LOSS_PER_SCENE` is the **largest candidate across the four methods**, identical in all four notebooks; the margin keeps the slowest method from meeting the target only where its curve flattens.

Note: as in the monocular study, the targets should apply the same rule to the **minimum** of each Protocol A test curve rather than to the final sample, because a method whose test curve peaks and then degrades would otherwise set the bar at a worse model than it can reach. The cell prints both.

In [ ]:
# 4.3  Protocol B calibration: prints this method's candidate targets only. Take the per-scene
# maximum over the four methods and paste it into all four notebooks.
import json
import os

print("%-18s %14s %14s %14s" % ("scene", "final eval L1", "min eval L1", "candidate x1.05"))
print("-" * 64)
candidates = {}
for _scene in N3DV_SCENES:
    _p = paths_for(_scene, "iterations")
    if not os.path.exists(_p["bench_json"]):
        print("%-18s %14s %14s %14s" % (_scene, "-", "-", "-"))
        continue
    with open(_p["bench_json"], "r") as handle:
        _summary = json.load(handle)
    _entries = _summary.get("entries") or []
    _final = _summary.get("final")
    if not _final or not _entries:
        print("%-18s %14s %14s %14s" % (_scene, "-", "-", "-"))
        continue
    _l1 = _final["eval_l1_loss"]
    _l1_min = min(e["eval_l1_loss"] for e in _entries)
    candidates[_scene] = round(_l1_min * 1.05, 6)
    print("%-18s %14.6f %14.6f %14.6f" % (_scene, _l1, _l1_min, candidates[_scene]))

print("\nThis method's candidates (compare with the other three, keep the max):")
print(json.dumps(candidates, indent=4))

### 4.4 Benchmark summary

Reads the JSON of the active scene. After a loop, call `activate_scene("<scene>")` to inspect another one.

In [ ]:
# 4.4  Benchmark summary for the active scene
import json
import os

import pandas as pd

train_duration = None
benchmark_summary = None
benchmark_table = None

if os.path.exists(BENCH_JSON_PATH):
    with open(BENCH_JSON_PATH, "r") as handle:
        benchmark_summary = json.load(handle)
    entries = benchmark_summary.get("entries", [])
else:
    entries = []

if entries:
    _all = pd.DataFrame(entries)
    columns = [c for c in ["iteration", "total_iterations", "images_seen",
                           "timesteps_done", "training_time_s", "psnr", "ssim", "lpips",
                           "eval_loss", "eval_l1_loss", "eval_photometric_loss",
                           "num_gaussians", "stage"] if c in _all.columns]
    benchmark_table = _all[columns]
    pd.set_option("display.float_format", lambda v: "%.5f" % v)

    print("Method   : %s" % benchmark_summary["method"])
    print("Scene    : %s" % benchmark_summary["scene"])
    print("Protocol : %s" % benchmark_summary["mode"])
    print("Status   : %s" % benchmark_summary["status"])
    print("Window   : %s frames, held-out camera %s"
          % (benchmark_summary.get("num_frames"),
             benchmark_summary.get("held_out_camera")))

    final = benchmark_summary["final"]
    train_duration = final["training_time_s"]

    if benchmark_summary["mode"] == "target_eval_loss":
        _cfg = benchmark_summary.get("config", {})
        _target = _cfg.get("target_eval_loss")
        print("Target   : eval_loss <= %.6f, held for %d consecutive samples"
              % (_target, _cfg.get("target_consecutive_hits", 2)))
        print("Reached  : %s" % benchmark_summary.get("target_reached"))
        # Report the first crossing of the target, not the termination (which includes the hysteresis).
        _first = next((e for e in entries if e["eval_loss"] <= _target), None)
        if _first is not None:
            print("\n  First crossing  : iteration %d, %.1f s (%.2f min)"
                  % (_first["iteration"], _first["training_time_s"],
                     _first["training_time_s"] / 60.0))
            print("  Stop confirmed  : iteration %d, %.1f s (%.2f min)"
                  % (final["iteration"], train_duration, train_duration / 60.0))
            print("  Hysteresis delay: %.1f s (+%.0f%%)"
                  % (train_duration - _first["training_time_s"],
                     100.0 * (train_duration / max(_first["training_time_s"], 1e-9) - 1.0)))
            print("  --> report the FIRST CROSSING as the cost of reaching the target.")
    print()
    display(benchmark_table)

    print()
    print("=" * 64)
    print(" FINAL POINT OF THE RUN")
    print("=" * 64)
    print("  Iterations                : %d" % final["iteration"])
    print("  Training time (net)       : %.2f s (%.2f min)"
          % (train_duration, train_duration / 60.0))
    print("  Benchmark overhead        : %.2f s" % final["benchmark_overhead_s"])
    print("  PSNR                      : %.4f dB" % final["psnr"])
    print("  SSIM                      : %.4f" % final["ssim"])
    print("  LPIPS                     : %.4f" % final["lpips"])
    print("  Eval loss (%-8s)      : %.6f" % (final["eval_loss_kind"], final["eval_loss"]))
    print("  Number of Gaussians       : %d" % final["num_gaussians"])
    print("  Test views evaluated      : %d  <-- must be %d across methods"
          % (final.get("num_eval_views", -1), NUM_FRAMES))
    if "eval_resolution" in final:
        print("  Evaluation resolution     : %d x %d  <-- must match across methods"
              % (final["eval_resolution"][1], final["eval_resolution"][0]))
    if "peak_vram_mb_nvidia_smi" in benchmark_summary:
        print("  Peak VRAM (device)        : %.0f MB"
              % benchmark_summary["peak_vram_mb_nvidia_smi"])
    print("=" * 64)
    print("JSON: %s" % BENCH_JSON_PATH)
else:
    print("No benchmark data for the active scene (%s, %s)." % (SCENE, TRAINING_MODE))

if benchmark_table is not None and len(benchmark_table) > 1:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    minutes = benchmark_table["training_time_s"] / 60.0
    axes[0].plot(minutes, benchmark_table["psnr"], marker="o", color="tab:blue")
    axes[0].set_xlabel("training time [min]"); axes[0].set_ylabel("PSNR [dB]")
    axes[0].set_title("PSNR (higher is better)")
    axes[1].plot(minutes, benchmark_table["ssim"], marker="o", color="tab:green")
    axes[1].set_xlabel("training time [min]"); axes[1].set_ylabel("SSIM")
    axes[1].set_title("SSIM (higher is better)")
    axes[2].plot(minutes, benchmark_table["lpips"], marker="s", color="tab:red")
    axes[2].set_xlabel("training time [min]"); axes[2].set_ylabel("LPIPS")
    axes[2].set_title("LPIPS (lower is better)")
    axes[3].plot(benchmark_table["iteration"], benchmark_table["eval_loss"],
                 marker="o", color="tab:orange")
    axes[3].set_xlabel("iteration"); axes[3].set_ylabel("evaluation loss")
    axes[3].set_title("Evaluation loss (lower is better)")
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.suptitle("%s -- %s (%s, %d frames)"
                 % (METHOD_NAME, SCENE, TRAINING_MODE, NUM_FRAMES))
    plt.tight_layout()
    plt.show()

---

# Part 2: evaluation and visualisation

**Do not run this part while a training loop is in progress.** Run cells 0.1 and 0.2 first (and the environment cells after a runtime restart), then call `activate_scene("<scene>")` to select the trained model. With `KEEP_MODEL_ON_DRIVE = False` the model exists only in the runtime that trained it, and with `DELETE_FRAMES_AFTER_TRAIN = True` the ground-truth frames were discarded too — re-run cell 3 for the scene before rendering.

The real-time WebSocket viewer of the monocular notebooks is not reproduced here: it adds nothing to the measurements and an N3DV model is too large to stream from a cloud runtime. Rendering, final metrics and an MP4 of the held-out view are kept.

---

## 5. Rendering and final metrics

The authors state that their evaluation code has not been released, so there is no upstream script to check the monitor against. This section renders the held-out camera from `params.npz` with the repository's own rasterizer and recomputes the metrics with the repository's own `calc_psnr`, `calc_ssim` and `l1_loss_v1` — the same definitions the monitor used, applied to the saved model instead of to the live one.

In [ ]:
# Render the held-out camera from params.npz and recompute the metrics.
import json
import os

import numpy as np
import torch
import lpips as lpips_pkg
from PIL import Image

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)
from helpers import setup_camera, params2rendervar
from external import calc_psnr, calc_ssim
from helpers import l1_loss_v1
from diff_gaussian_rasterization import GaussianRasterizer as Renderer

params_path = os.path.join(MODEL_DIR, "params.npz")
meta_path = os.path.join(source_path_for(SCENE), "test_meta.json")

if not (os.path.exists(params_path) and os.path.exists(meta_path)):
    print("Missing %s or %s: re-run cell 3 for this scene and train it first."
          % (params_path, meta_path))
else:
    blob = dict(np.load(params_path))
    meta = json.load(open(meta_path))
    n_frames = min(len(meta["fn"]), blob["means3D"].shape[0])
    model = lpips_pkg.LPIPS(net=LPIPS_NET).cuda().eval()

    out_dir = os.path.join(MODEL_DIR, "test_renders")
    os.makedirs(out_dir, exist_ok=True)
    acc = {"psnr": 0.0, "ssim": 0.0, "lpips": 0.0, "l1": 0.0}
    with torch.no_grad():
        for t in range(n_frames):
            params = {}
            for key, value in blob.items():
                array = value[t] if value.ndim and value.shape[0] == n_frames else value
                params[key] = torch.tensor(array).cuda().float()
            camera = setup_camera(meta["w"], meta["h"], meta["k"][t][0],
                                  meta["w2c"][t][0], near=1.0, far=100)
            image, _, _ = Renderer(raster_settings=camera)(**params2rendervar(params))
            image = torch.clamp(image, 0.0, 1.0)
            gt_arr = np.array(Image.open(os.path.join(
                source_path_for(SCENE), "ims", meta["fn"][t][0])))
            gt = torch.tensor(gt_arr).permute(2, 0, 1).cuda().float() / 255.0
            acc["psnr"] += calc_psnr(image[None], gt[None]).mean().item()
            acc["ssim"] += calc_ssim(image, gt).mean().item()
            acc["l1"] += l1_loss_v1(image, gt).item()
            acc["lpips"] += model(image[None] * 2 - 1, gt[None] * 2 - 1).mean().item()
            Image.fromarray((image.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                            ).save(os.path.join(out_dir, "%05d.png" % t))
    for k in ("psnr", "ssim", "lpips", "l1"):
        print("  %-6s %.6f" % (k, acc[k] / n_frames))
    print("\nRendered %d frames into %s" % (n_frames, out_dir))

## 6. Video export of the held-out view

Assembles the rendered frames of the held-out camera into an MP4, next to the ground truth.

In [ ]:
# Encode the rendered test frames into a side-by-side MP4 (render | ground truth)
import glob
import os
import shutil

FPS_OUT = 30
renders = sorted(glob.glob(os.path.join(MODEL_DIR, "test_renders", "*.png")))
print("%d rendered frames" % len(renders))

if renders:
    gt_dir = os.path.join(MODEL_DIR, "test_gt")
    os.makedirs(gt_dir, exist_ok=True)
    src_dir = os.path.join(source_path_for(SCENE), "ims", HELD_OUT_CAMERA)
    for t in range(len(renders)):
        source = os.path.join(src_dir, "%d.png" % t)
        if os.path.exists(source):
            shutil.copyfile(source, os.path.join(gt_dir, "%05d.png" % t))

    out = os.path.join(MODEL_DIR, "%s_test.mp4" % RUN_NAME)
    render_glob = os.path.join(os.path.dirname(renders[0]), "%05d.png")
    gt_glob = os.path.join(gt_dir, "%05d.png")
    !ffmpeg -y -loglevel error -framerate {FPS_OUT} -i "{render_glob}" -framerate {FPS_OUT} -i "{gt_glob}" \
        -filter_complex hstack -c:v libx264 -pix_fmt yuv420p -crf 18 "{out}"
    print("written: %s" % out)

    from IPython.display import HTML
    import base64
    with open(out, "rb") as handle:
        payload = base64.b64encode(handle.read()).decode()
    display(HTML('<video width=960 controls><source src="data:video/mp4;base64,%s"></video>'
                 % payload))
else:
    print("Nothing to encode: run section 5 first.")